# PASS 5 v10.1.2 — Budgeted, Calibrated Tree Baselines for Temporal QUIC Classification

**Goal:** replace the failed Version 9 final-fit schedule with a reproducible, validation-only, time-bounded comparison of LightGBM multiclass and XGBoost one-vs-rest (OvR), while retaining the complete June 2024–March 2025 training population and complete May 2025 test population.

## What changed after the Version 9 timeout
- HPO has a **15-minute wall-clock target per tree family**, identical validation partitions and the same fixed pilot set; actual time and number of completed trials are recorded. LightGBM and XGBoost use model-appropriate search spaces, but both optimize April macro-F1.
- Final LightGBM rounds are capped at 350 (derived from the best pilot iteration); XGBoost OvR rounds are capped at 220 per class (derived from its HPO best-round distribution). Both still have April early stopping; cap hits are explicitly reported, never described as convergence.
- The full 600,000-flow April reservoir is split once into disjoint 30k HPO, 70k early-stopping, 400k stacker-fit and 100k calibration-audit partitions. Both tree families use the same partitions and the same multinomial logistic-regression stacker recipe trained on the 400k April partition. The 100k audit split is not used to select the model or stacker.
- May is not read until both base models and their stackers are frozen. Final uncertainty intervals use a paired **day-block bootstrap** for the two tree models. No neural bootstrap intervals are fabricated from an aggregate confusion matrix.
- Large temporary matrices, XGBoost external-memory cache, and periodic class checkpoints are moved to `/kaggle/temp` (or `/tmp` fallback), reducing saved-output size. Completed class boosters and the progress ledger stay in the run output. XGBoost checkpoint interval is 25 rounds rather than 5. If a full run times out and Kaggle preserves `/kaggle/working`, set `RESUME_EXISTING_RUN=True` and run all cells again; data preparation and HPO repeat, while completed class boosters are reused only if the fitted configuration signature matches.
- The prior `NameError` pattern from deleting an undefined temporary variable is avoided: temporary cleanup uses guarded `globals().pop(name, None)`.

## Important scientific boundary
The Pass 5 tree-family comparison is much fairer: same features, complete train population, class-weight policy, validation partitions, HPO wall-clock target, April-label/calibration policy and May test population. **This notebook alone cannot make the historical neural result a controlled same-pipeline comparison.** The frozen `stacking_A_C` reference is kept as an external point-estimate row only; its provenance/metric discrepancy with Pass 3 must be reconciled and the neural pipeline retrained in the same protocol before claiming architectural superiority. Neural confidence intervals and neural-vs-tree significance claims are intentionally omitted.

Run on Kaggle: attach the CESNET-QUICEXT-25 M dataset and `finaljson` reference-artifact dataset, then **Restart Session → Run All** with `DRY_RUN=False` and `RESUME_EXISTING_RUN=False` for a clean run. After a timeout, if `/kaggle/working/pass5_fair_full_run` still contains completed class models and its progress ledger, set `RESUME_EXISTING_RUN=True` before restarting. Set `DRY_RUN=True` only for a diagnostic; dry-run metrics are not paper-valid.

### v10.1.2 reliability correction (LightGBM split failure)

The previous full run stopped during the LightGBM fit with `best_split_info.left_count > 0` after the GPU trainer had started fitting the full 27.7M-row training matrix. The notebook now enables double-precision GPU histogram accumulation for the primary fit and uses a bounded retry sequence if the native LightGBM trainer still fails: (1) the selected HPO configuration on GPU with double precision, (2) a conservative GPU configuration with fewer bins / leaves and a larger minimum leaf size, then (3) the same conservative model on CPU with column-wise histogram construction. Each attempt and its exception are written to the run log and `lightgbm_training_attempts.json`; the final model/config records the backend and any robustness overrides. This is an execution-reliability fallback, not a claim that the retry configuration is HPO-optimal. The May test remains untouched until all validation-only model/stacker decisions are frozen.


In [ ]:
# ============================================================
# Cell 0 — RUN SWITCH + EXPERIMENT CONFIGURATION
# ============================================================

DRY_RUN = False  # set True only for a diagnostic run; results are not paper-valid
RESUME_EXISTING_RUN = False  # set True only to preserve a timed-out v10 RUN_DIR and reuse completed XGBoost class models

SEED = 20261003
VAL_SEED = SEED + 1
TEST_BENCHMARK_SEED = SEED + 2
BOOTSTRAP_SEED = SEED + 3
BOOTSTRAP_REPS = 1000

DATASET_NAME = "CESNET-QUICEXT-25"
DATASET_SIZE = "M"
N_CLASSES_EXPECTED = 50
N_FEATURES_EXPECTED = 101
TRAIN_EXPECTED_ROWS = 27_727_267
VAL_SAMPLES_FULL = 600_000  # disjoint April partitions: 30k + 70k + 400k + 100k
TEST_EXPECTED_ROWS = 2_359_061
PPI_PACKETS = 30
PPI_CHANNELS = 3

# Data preparation / external-memory controls
READ_CHUNK = 250_000
PREPROCESS_CHUNK = 1_000_000
XGB_ITERATOR_ROWS = 100_000
XGB_CACHE_HOST_RATIO = 1.0
XGB_GPU_HEADROOM_GIB = 1.5
XGB_CHECKPOINT_INTERVAL = 25

# HPO uses one shared, fixed, 100k class-balanced training pilot (2k/class),
# one shared 30k April HPO validation partition, and a 15-minute wall-clock
# target per model family. Trial count is not forced to match if one family
# reaches its wall-clock budget sooner; actual trials and seconds are logged.
HPO_PILOT_PER_CLASS = 2_000
HPO_PILOT_ROWS = HPO_PILOT_PER_CLASS * 50
HPO_VALIDATION_ROWS = 30_000
HPO_BUDGET_SECONDS_PER_FAMILY = 900
HPO_LGBM_MAX_ROUNDS = 240
HPO_LGBM_EARLY_STOPPING = 30
HPO_XGB_MAX_ROUNDS = 140
HPO_XGB_EARLY_STOPPING = 20

# Final refits are deliberately bounded to avoid another 12-hour timeout.
# The limits are not treated as convergence: cap-hit diagnostics are persisted.
FULL_LGBM_MAX_ROUNDS = 350
FULL_LGBM_EARLY_STOPPING = 35
FULL_XGB_MAX_ROUNDS = 220
FULL_XGB_EARLY_STOPPING = 25
FULL_XGB_MAX_BIN = 64

DRY_TRAIN_PER_CLASS = 50
DRY_VAL_PER_CLASS = 20
DRY_TEST_PER_CLASS = 110
DRY_LGBM_ESTIMATORS = 20
DRY_XGB_ESTIMATORS = 20
DRY_EARLY_STOPPING = 5
DRY_BENCHMARK_BATCHES = 5

print("=" * 80)
print("PASS 5 v10 — FAIR, BUDGETED TREE-BASELINE COMPARISON")
print("DRY_RUN =", DRY_RUN)
print("HPO target per family =", HPO_BUDGET_SECONDS_PER_FAMILY, "seconds")
print("Final round caps: LightGBM =", FULL_LGBM_MAX_ROUNDS,
      "; XGBoost OvR =", FULL_XGB_MAX_ROUNDS)
print("=" * 80)

In [ ]:
# ============================================================
# Cell 1 — IMPORTS, PATHS, LOGGING, HARDWARE
# ============================================================

import os
import gc
import sys
import json
import time
import shutil
import subprocess
import warnings
import importlib.metadata as importlib_metadata
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import tables as tb
import lightgbm as lgb
import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
np.random.seed(SEED)

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

def resolve_input_path(preferred, filename):
    if preferred.exists():
        return preferred
    matches = sorted(
        p for p in KAGGLE_INPUT.rglob(filename)
        if p.is_file()
    )
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {KAGGLE_INPUT}."
        )
    raise RuntimeError(
        f"Multiple candidates found for {filename}:\n"
        + "\n".join(map(str, matches))
    )

DB_PATH = resolve_input_path(
    Path(
        "/kaggle/input/datasets/yahiaa1/"
        "cesnet-quixext-25-m/CESNET-QUICEXT-25/M/"
        "CESNET-QUICEXT-25-M.h5"
    ),
    "CESNET-QUICEXT-25-M.h5",
)

SERVICEMAP_PATH = resolve_input_path(
    DB_PATH.parent / "servicemap.csv",
    "servicemap.csv",
)

NEURAL_METRICS_PATH = resolve_input_path(
    Path(
        "/kaggle/input/datasets/yahiaa1/finaljson/"
        "final_test_metrics.json"
    ),
    "final_test_metrics.json",
)

NEURAL_CM_PATH = resolve_input_path(
    Path(
        "/kaggle/input/datasets/yahiaa1/finaljson/"
        "final_confusion_matrix.npy"
    ),
    "final_confusion_matrix.npy",
)

NEURAL_BENCHMARK_PATH = resolve_input_path(
    Path(
        "/kaggle/input/datasets/yahiaa1/finaljson/"
        "final_inference_benchmark.json"
    ),
    "final_inference_benchmark.json",
)

RUN_DIR = KAGGLE_WORKING / (
    "pass5_fair_dry_run" if DRY_RUN else "pass5_fair_full_run"
)

if RUN_DIR.exists() and not RESUME_EXISTING_RUN:
    shutil.rmtree(RUN_DIR)
elif RUN_DIR.exists() and RESUME_EXISTING_RUN:
    print("RESUME_EXISTING_RUN=True: preserving the existing run directory and completed class-model ledger.")
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Keep the large memmaps and XGBoost cache out of Kaggle's saved output bundle.
SCRATCH_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
SCRATCH_DIR = SCRATCH_ROOT / ("pass5_fair_dry_scratch" if DRY_RUN else "pass5_fair_full_scratch")
if SCRATCH_DIR.exists():
    shutil.rmtree(SCRATCH_DIR)
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
PASS5_RUN_START_EPOCH = time.time()

print("Run artifacts:", RUN_DIR)
print("Large temporary arrays/cache:", SCRATCH_DIR)
print("Working free disk:", f"{shutil.disk_usage(KAGGLE_WORKING).free / (1024**3):.2f} GiB")
print("Scratch free disk:", f"{shutil.disk_usage(SCRATCH_DIR).free / (1024**3):.2f} GiB")

LOG_PATH = RUN_DIR / "pass5_run.log"
LOG_FILE = open(
    LOG_PATH,
    "a" if RESUME_EXISTING_RUN else "w",
    encoding="utf-8",
    buffering=1,
)

# IMPORTANT: do not replace sys.stdout/sys.stderr in a Jupyter/Kaggle notebook.
# A persistent wrapper around the kernel OutStream can retain the parent cell
# execution context and cause later-cell output to appear in the first cell.
# tee_print writes to the CURRENT cell's stdout and mirrors notebook-owned
# print() calls to the run log.
_ORIGINAL_PRINT = print

def tee_print(*args, **kwargs):
    # Always use the CURRENT Jupyter stdout. This preserves per-cell output.
    _ORIGINAL_PRINT(*args, **kwargs)

    # Mirror notebook-owned print() calls to the run log when available.
    # The guard makes rerunning cells after the final log close harmless.
    if kwargs.get("file") is None:
        log_file = globals().get("LOG_FILE")
        if log_file is not None and not getattr(log_file, "closed", True):
            log_kwargs = {
                "sep": kwargs.get("sep", " "),
                "end": kwargs.get("end", "\n"),
                "flush": True,
            }
            _ORIGINAL_PRINT(*args, file=log_file, **log_kwargs)

with open(
    NEURAL_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    NEURAL_METRICS = json.load(f)

NEURAL_CM = np.load(
    NEURAL_CM_PATH
)

with open(
    NEURAL_BENCHMARK_PATH,
    "r",
    encoding="utf-8",
) as f:
    NEURAL_BENCHMARK = json.load(f)

# ------------------------------------------------------------
# Frozen neural-reference values used throughout Pass 5.
# ------------------------------------------------------------

NEURAL_SELECTED_METHOD = str(
    NEURAL_METRICS["selected_method"]
)

NEURAL_TEST_FLOWS = int(
    NEURAL_METRICS["test_flows"]
)

NEURAL_ACCURACY = float(
    NEURAL_METRICS["accuracy"]
)

NEURAL_MACRO_PRECISION = float(
    NEURAL_METRICS["precision_macro"]
)

NEURAL_MACRO_RECALL = float(
    NEURAL_METRICS["recall_macro"]
)

NEURAL_MACRO_F1 = float(
    NEURAL_METRICS["f1_macro"]
)

NEURAL_WEIGHTED_F1 = float(
    NEURAL_METRICS["f1_weighted"]
)

NEURAL_ECE = float(
    NEURAL_METRICS["ece_10bin"]
)

assert NEURAL_CM.shape == (
    N_CLASSES_EXPECTED,
    N_CLASSES_EXPECTED,
)

assert int(
    NEURAL_CM.sum()
) == NEURAL_TEST_FLOWS

assert (
    NEURAL_TEST_FLOWS
    == TEST_EXPECTED_ROWS
)

assert NEURAL_CM.shape == (50, 50)
assert int(NEURAL_CM.sum()) == int(NEURAL_METRICS["test_flows"])

working_free_disk_gib = shutil.disk_usage(KAGGLE_WORKING).free / (1024**3)
scratch_free_disk_gib = shutil.disk_usage(SCRATCH_DIR).free / (1024**3)
free_disk_gib = min(working_free_disk_gib, scratch_free_disk_gib)
total_ram_gib = (
    os.sysconf("SC_PAGE_SIZE")
    * os.sysconf("SC_PHYS_PAGES")
    / (1024**3)
)

GPU_VISIBLE = False
GPU_INFO = ""
try:
    r = subprocess.run(
        ["nvidia-smi", "-L"],
        capture_output=True,
        text=True,
        timeout=10,
    )
    GPU_VISIBLE = r.returncode == 0 and bool(r.stdout.strip())
    GPU_INFO = r.stdout.strip()
except Exception as exc:
    GPU_INFO = repr(exc)

CUPY_AVAILABLE = False
cp = None
try:
    import cupy as cp
    CUPY_AVAILABLE = True
except Exception:
    pass

# XGBoost external-memory GPU training requires CUDA async allocation for
# both XGBoost and the CuPy arrays passed through its DataIter.
CUPY_ASYNC_POOL_AVAILABLE = False
CUPY_ASYNC_POOL = None
if CUPY_AVAILABLE:
    try:
        CUPY_ASYNC_POOL = cp.cuda.MemoryAsyncPool()
        cp.cuda.set_allocator(CUPY_ASYNC_POOL.malloc)
        CUPY_ASYNC_POOL_AVAILABLE = True
    except Exception as exc:
        tee_print(
            "CuPy async memory pool unavailable:",
            repr(exc),
        )

# Fail fast BEFORE HDF5 traversal/full-matrix construction if either the
# LightGBM OpenCL GPU trainer or XGBoost's required CUDA path is unavailable.
LGBM_GPU_PREFLIGHT_OK = False
if not DRY_RUN:
    if not GPU_VISIBLE:
        raise RuntimeError(
            "Full run requires a visible NVIDIA GPU; select Kaggle GPU T4 x2."
        )
    if not CUPY_AVAILABLE:
        raise RuntimeError(
            "Full run requires CuPy for XGBoost GPU external-memory training."
        )
    if not CUPY_ASYNC_POOL_AVAILABLE:
        raise RuntimeError(
            "Full run requires cp.cuda.MemoryAsyncPool. The notebook stopped before data preparation."
        )

    _probe_rng = np.random.default_rng(SEED + 501)
    _probe_y = np.tile(
        np.arange(N_CLASSES_EXPECTED, dtype=np.int32), 20
    )
    _probe_X = _probe_rng.normal(
        size=(len(_probe_y), N_FEATURES_EXPECTED)
    ).astype(np.float32)
    # Make one feature predictive so the probe executes actual GPU split work.
    _probe_X[:, 0] = (
        _probe_y.astype(np.float32)
        + _probe_rng.normal(0.0, 0.05, size=len(_probe_y)).astype(np.float32)
    )
    _probe_order = _probe_rng.permutation(len(_probe_y))
    _probe_X = _probe_X[_probe_order]
    _probe_y = _probe_y[_probe_order]
    _probe_lgb = LGBMClassifier(
        objective="multiclass",
        num_class=N_CLASSES_EXPECTED,
        n_estimators=2,
        learning_rate=0.1,
        num_leaves=7,
        max_depth=-1,
        min_child_samples=1,
        max_bin=63,
        device_type="gpu",
        gpu_device_id=0,
        gpu_use_dp=False,
        random_state=SEED,
        n_jobs=2,
        verbosity=1,
    )
    try:
        tee_print("Running fail-fast LightGBM OpenCL GPU probe...")
        _probe_lgb.fit(_probe_X, _probe_y)
        LGBM_GPU_PREFLIGHT_OK = True
        tee_print("LightGBM OpenCL GPU probe: PASSED")
    except Exception as exc:
        raise RuntimeError(
            "LightGBM GPU probe failed. The run is stopped before reading/building the full training matrix. "
            "The next full run must use a LightGBM build with OpenCL GPU support."
        ) from exc
    finally:
        try:
            del _probe_lgb, _probe_X, _probe_y, _probe_order, _probe_rng
        except Exception:
            pass
        gc.collect()

VERSIONS = {}
for name in [
    "numpy",
    "pandas",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "joblib",
    "tables",
]:
    try:
        VERSIONS[name] = importlib_metadata.version(name)
    except Exception:
        VERSIONS[name] = "unknown"

TRAIN_MATRIX_GIB = (
    TRAIN_EXPECTED_ROWS
    * N_FEATURES_EXPECTED
    * 4
    / (1024**3)
)
VAL_MATRIX_GIB = (
    VAL_SAMPLES_FULL
    * N_FEATURES_EXPECTED
    * 4
    / (1024**3)
)
MIN_FREE_DISK_GIB = (
    TRAIN_MATRIX_GIB
    + VAL_MATRIX_GIB
    + 5.0
)

tee_print("=" * 80)
tee_print("PASS 5 ENVIRONMENT")
tee_print("=" * 80)
tee_print("DRY_RUN:", DRY_RUN)
tee_print("Database:", DB_PATH)
tee_print("Database size:", f"{DB_PATH.stat().st_size/(1024**3):.2f} GiB")
tee_print("Working disk:", f"{working_free_disk_gib:.2f} GiB free")
tee_print("Scratch disk:", f"{scratch_free_disk_gib:.2f} GiB free")
tee_print("Conservative local free-disk check:", f"{free_disk_gib:.2f} GiB free")
tee_print("System RAM:", f"{total_ram_gib:.2f} GiB")
tee_print("GPU visible:", GPU_VISIBLE)
tee_print("CuPy available:", CUPY_AVAILABLE)
tee_print(GPU_INFO)

tee_print("\nNeural reference:")
tee_print("  method:", NEURAL_METRICS["selected_method"])
tee_print("  test flows:", NEURAL_METRICS["test_flows"])
tee_print("  accuracy:", f"{NEURAL_METRICS['accuracy']:.8f}")
tee_print("  macro F1:", f"{NEURAL_METRICS['f1_macro']:.8f}")
tee_print("  weighted F1:", f"{NEURAL_METRICS['f1_weighted']:.8f}")
tee_print("  ECE:", f"{NEURAL_METRICS['ece_10bin']:.8f}")
tee_print("  latency:", f"{NEURAL_BENCHMARK['latency_ms_per_batch']:.3f} ms/batch")
tee_print("  throughput:", f"{NEURAL_BENCHMARK['throughput_flows_per_sec']:.2f} flows/s")

tee_print("\nPackage versions:")
for name, version in VERSIONS.items():
    tee_print(f"  {name:<16}: {version}")

tee_print("\nFull-run storage estimate:")
tee_print("  X_train:", f"{TRAIN_MATRIX_GIB:.2f} GiB")
tee_print("  X_val  :", f"{VAL_MATRIX_GIB:.2f} GiB")
tee_print("  target :", f"{MIN_FREE_DISK_GIB:.2f} GiB free")

if not DRY_RUN:
    if free_disk_gib < MIN_FREE_DISK_GIB:
        raise RuntimeError(
            "Insufficient Kaggle working-disk space for the full run."
        )
    if total_ram_gib < 24.0:
        raise RuntimeError(
            "Host RAM below 24 GiB; refusing the full LightGBM run."
        )
    if not GPU_VISIBLE:
        raise RuntimeError(
            "Full run requires a visible NVIDIA GPU."
        )
    if not CUPY_AVAILABLE:
        raise RuntimeError(
            "Full XGBoost external-memory path requires CuPy."
        )
    if not CUPY_ASYNC_POOL_AVAILABLE:
        raise RuntimeError(
            "Full XGBoost external-memory path requires the CuPy CUDA async allocator."
        )
    if not LGBM_GPU_PREFLIGHT_OK:
        raise RuntimeError(
            "LightGBM GPU preflight did not complete successfully."
        )

tee_print("\nEnvironment validation complete.")


In [ ]:
# ============================================================
# Cell 2 — PYTABLES STRUCTURE + EXACT 50-CLASS LABEL SPACE
# ============================================================

tee_print("=" * 80)
tee_print("HDF5 / PYTABLES STRUCTURE")
tee_print("=" * 80)

PPI_COLUMN = "PPI"
APP_COLUMN = "APP"

AUX_FEATURES = [
    "BYTES",
    "BYTES_REV",
    "PACKETS",
    "PACKETS_REV",
    "DURATION",
    "FLOW_ENDREASON_ACTIVE",
    "FLOW_ENDREASON_IDLE",
    "FLOW_ENDREASON_OTHER",
    "PPI_DURATION",
    "PPI_LEN",
    "PPI_ROUNDTRIPS",
]

N_FEATURES = (
    PPI_PACKETS * PPI_CHANNELS
    + len(AUX_FEATURES)
)

assert N_FEATURES == 101

servicemap = pd.read_csv(
    SERVICEMAP_PATH,
    index_col="Tag",
)

SERVICEMAP_LABELS = {
    str(x) for x in servicemap.index.tolist()
}

with tb.open_file(str(DB_PATH), mode="r") as database:
    assert database.__contains__("/flows")

    flows_group = database.get_node("/flows")

    flow_tables = [
        node
        for node in flows_group._v_children.values()
        if isinstance(node, tb.Table)
    ]

    FLOW_KEYS = sorted(
        table._v_pathname
        for table in flow_tables
    )

    assert FLOW_KEYS

    FLOW_KEYS_BY_DATE = {}

    for key in FLOW_KEYS:
        name = Path(key).name
        if (
            name.startswith("D")
            and len(name) == 9
            and name[1:].isdigit()
        ):
            FLOW_KEYS_BY_DATE[name[1:]] = key

    AVAILABLE_DATES = sorted(
        FLOW_KEYS_BY_DATE
    )

    assert len(AVAILABLE_DATES) == 360

    first_table = database.get_node(FLOW_KEYS[0])

    TABLE_COLUMNS = list(
        first_table.dtype.names
    )

    missing = [
        c
        for c in [
            PPI_COLUMN,
            *AUX_FEATURES,
            APP_COLUMN,
        ]
        if c not in TABLE_COLUMNS
    ]

    assert not missing, (
        f"Missing fields: {missing}"
    )

    enum_dict = dict(
        first_table.get_enum(APP_COLUMN)
    )

    keys_numeric = all(
        isinstance(k, (int, np.integer))
        for k in enum_dict
    )

    values_numeric = all(
        isinstance(v, (int, np.integer))
        for v in enum_dict.values()
    )

    if keys_numeric:
        APP_CODE_TO_NAME = {
            int(k): str(v)
            for k, v in enum_dict.items()
        }
        ENUM_ORIENTATION = "code -> name"
    elif values_numeric:
        APP_CODE_TO_NAME = {
            int(v): str(k)
            for k, v in enum_dict.items()
        }
        ENUM_ORIENTATION = "name -> code"
    else:
        raise TypeError(
            "Unsupported APP enum representation."
        )

    candidate_labels = (
        set(APP_CODE_TO_NAME.values())
        .intersection(SERVICEMAP_LABELS)
    )

    BACKGROUND_LABELS = sorted(
        x for x in candidate_labels
        if x.endswith("-background")
    )

    KNOWN_APPS = sorted(
        x for x in candidate_labels
        if not x.endswith("-background")
    )

    N_CLASSES = len(KNOWN_APPS)

    assert N_CLASSES == 50, (
        f"Expected 50 classes, got {N_CLASSES}: {KNOWN_APPS}"
    )

    CLASS_TO_ID = {
        name: i
        for i, name in enumerate(KNOWN_APPS)
    }

    ID_TO_CLASS = {
        i: name
        for name, i in CLASS_TO_ID.items()
    }

    NAME_TO_APP_CODE = {
        name: code
        for code, name
        in APP_CODE_TO_NAME.items()
    }

    MAX_APP_CODE = max(APP_CODE_TO_NAME)

    APP_CODE_TO_CLASS_ID = np.full(
        MAX_APP_CODE + 1,
        -1,
        dtype=np.int16,
    )

    for class_id, name in enumerate(KNOWN_APPS):
        APP_CODE_TO_CLASS_ID[
            NAME_TO_APP_CODE[name]
        ] = class_id

    KNOWN_APP_CODES = np.array(
        [
            NAME_TO_APP_CODE[name]
            for name in KNOWN_APPS
        ],
        dtype=np.int16,
    )

    TOTAL_DATABASE_ROWS = sum(
        int(database.get_node(key).nrows)
        for key in FLOW_KEYS
    )

TRAIN_DATES = [
    d for d in AVAILABLE_DATES
    if "20240601" <= d <= "20250331"
]
VAL_DATES = [
    d for d in AVAILABLE_DATES
    if "20250401" <= d <= "20250430"
]
TEST_DATES = [
    d for d in AVAILABLE_DATES
    if "20250501" <= d <= "20250531"
]

assert len(TRAIN_DATES) == 299
assert len(VAL_DATES) == 30
assert len(TEST_DATES) == 31

tee_print("Flow tables:", len(FLOW_KEYS))
tee_print("Database rows:", f"{TOTAL_DATABASE_ROWS:,}")
tee_print("Date range:", AVAILABLE_DATES[0], "->", AVAILABLE_DATES[-1])
tee_print("APP enum entries:", len(APP_CODE_TO_NAME))
tee_print("APP enum orientation:", ENUM_ORIENTATION)
tee_print("Excluded background:", BACKGROUND_LABELS)
tee_print("Known classes:", N_CLASSES)
tee_print("Features:", N_FEATURES)
tee_print("TRAIN:", TRAIN_DATES[0], "->", TRAIN_DATES[-1], len(TRAIN_DATES), "days")
tee_print("VAL:", VAL_DATES[0], "->", VAL_DATES[-1], len(VAL_DATES), "days")
tee_print("TEST:", TEST_DATES[0], "->", TEST_DATES[-1], len(TEST_DATES), "days")

tee_print("\nStructural validation complete.")
tee_print("TEST DATA HAS NOT BEEN READ.")


In [ ]:
# ============================================================
# Cell 3 — NATIVE PYTABLES READER
# ============================================================

FEATURE_COLUMNS = (
    [f"IPT_{i}" for i in range(1, 31)]
    + [f"DIR_{i}" for i in range(1, 31)]
    + [f"SIZE_{i}" for i in range(1, 31)]
    + AUX_FEATURES
)

assert len(FEATURE_COLUMNS) == 101

def raw_to_features(raw):
    if (
        not isinstance(raw, np.ndarray)
        or raw.dtype.names is None
    ):
        raise TypeError(
            "Expected a structured NumPy array."
        )

    n = len(raw)

    if n == 0:
        return (
            np.empty((0, 101), dtype=np.float32),
            np.empty((0,), dtype=np.int16),
        )

    ppi = np.asarray(
        raw[PPI_COLUMN],
        dtype=np.float32,
    )

    if ppi.shape[1:] == (3, 30):
        ppi = np.transpose(
            ppi,
            (0, 2, 1),
        )
    elif ppi.shape[1:] != (30, 3):
        raise ValueError(
            f"Unexpected PPI shape: {ppi.shape}"
        )

    X = np.empty(
        (n, 101),
        dtype=np.float32,
    )

    X[:, 0:30] = ppi[:, :, 0]
    X[:, 30:60] = ppi[:, :, 1]
    X[:, 60:90] = ppi[:, :, 2]

    for j, feature in enumerate(AUX_FEATURES):
        X[:, 90 + j] = np.asarray(
            raw[feature],
            dtype=np.float32,
        ).reshape(n)

    app_codes = np.asarray(
        raw[APP_COLUMN],
        dtype=np.int16,
    )

    if (
        app_codes.min(initial=0) < 0
        or app_codes.max(initial=0)
        >= len(APP_CODE_TO_CLASS_ID)
    ):
        raise ValueError(
            "APP code outside lookup range."
        )

    y = APP_CODE_TO_CLASS_ID[app_codes]

    if np.any(y < 0):
        raise ValueError(
            "Unknown/background APP code passed through reader."
        )

    return X, y


def iter_period_chunks(
    dates,
    read_chunk=READ_CHUNK,
):
    with tb.open_file(
        str(DB_PATH),
        mode="r",
    ) as database:

        for date_str in dates:

            key = FLOW_KEYS_BY_DATE[
                date_str
            ]

            table = database.get_node(
                key
            )

            nrows = int(
                table.nrows
            )

            tee_print(
                f"[READ] {key} | rows={nrows:,}"
            )

            for start in range(
                0,
                nrows,
                read_chunk,
            ):

                stop = min(
                    start + read_chunk,
                    nrows,
                )

                # Read each structured chunk exactly once. The previous
                # reader first fetched APP, then re-read the full chunk or
                # performed random coordinate reads, multiplying HDF5 I/O.
                raw = table.read(
                    start=start,
                    stop=stop,
                )

                if len(raw) == 0:
                    del raw
                    continue

                app_codes = np.asarray(
                    raw[APP_COLUMN],
                    dtype=np.int16,
                )
                known_mask = np.isin(
                    app_codes,
                    KNOWN_APP_CODES,
                )

                known_count = int(
                    known_mask.sum()
                )

                if known_count == 0:
                    del app_codes
                    del known_mask
                    del raw
                    continue

                if known_count != len(raw):
                    raw = raw[known_mask]

                yield raw

                del app_codes
                del known_mask
                del raw

tee_print("Native PyTables reader ready.")
tee_print("No pd.HDFStore access exists in Pass 5.")


In [ ]:
# ============================================================
# Cell 4 — READER SMOKE TEST
# ============================================================

tee_print("=" * 80)
tee_print("READER SMOKE TEST")
tee_print("=" * 80)

raw_smoke = next(
    iter_period_chunks(
        [TRAIN_DATES[0]]
    )
)

X_smoke, y_smoke = raw_to_features(
    raw_smoke
)

tee_print("Raw known rows:", len(raw_smoke))
tee_print("X:", X_smoke.shape, X_smoke.dtype)
tee_print("y:", y_smoke.shape, y_smoke.dtype)
tee_print(
    "Label range:",
    int(y_smoke.min()),
    "->",
    int(y_smoke.max()),
)
tee_print(
    "Classes present:",
    len(np.unique(y_smoke))
)
tee_print(
    "Finite:",
    bool(np.isfinite(X_smoke).all())
)
tee_print(
    "DIR unique:",
    np.unique(
        X_smoke[:, 30:60]
    )
)

assert X_smoke.shape == (
    len(y_smoke),
    101,
)

assert np.isfinite(
    X_smoke
).all()

assert np.all(
    np.isin(
        np.unique(
            X_smoke[:, 30:60]
        ),
        [-1.0, 0.0, 1.0],
    )
)

assert np.all(
    (
        y_smoke >= 0
    )
    &
    (
        y_smoke < 50
    )
)

del raw_smoke, X_smoke, y_smoke
gc.collect()

tee_print("\nSMOKE TEST PASSED.")
tee_print("May/test data have NOT been accessed.")


In [ ]:
# ============================================================
# Cell 5 — DETERMINISTIC SAMPLING UTILITIES
# ============================================================

def reservoir_sample_period(
    dates,
    n_samples,
    seed,
    label,
):
    rng = np.random.default_rng(seed)

    if n_samples <= 0:
        raise ValueError("n_samples must be positive.")

    X_res = np.empty(
        (n_samples, N_FEATURES),
        dtype=np.float32,
    )
    y_res = np.empty(
        (n_samples,),
        dtype=np.int16,
    )

    filled = 0
    seen = 0
    t0 = time.time()

    for raw in iter_period_chunks(dates):

        X_chunk, y_chunk = raw_to_features(raw)
        m = len(y_chunk)

        if m == 0:
            continue

        previous_seen = seen

        if filled < n_samples:

            take = min(
                n_samples - filled,
                m,
            )

            X_res[
                filled:filled + take
            ] = X_chunk[:take]

            y_res[
                filled:filled + take
            ] = y_chunk[:take]

            filled += take

        else:
            take = 0

        if (
            filled == n_samples
            and take < m
        ):

            remaining = m - take

            positions = (
                previous_seen
                + np.arange(
                    take + 1,
                    m + 1,
                    dtype=np.int64,
                )
            )

            j = (
                rng.random(
                    remaining
                )
                * positions
            ).astype(
                np.int64
            )

            replace = j < n_samples

            if np.any(replace):

                source_idx = (
                    take
                    + np.flatnonzero(
                        replace
                    )
                )

                target_idx = (
                    j[replace]
                )

                X_res[
                    target_idx
                ] = X_chunk[
                    source_idx
                ]

                y_res[
                    target_idx
                ] = y_chunk[
                    source_idx
                ]

        seen += m

        del X_chunk, y_chunk, raw

        if (
            seen // 1_000_000
            !=
            (
                seen - m
            ) // 1_000_000
        ):
            tee_print(
                f"[{label}] seen={seen:,} "
                f"elapsed={(time.time()-t0)/60:.2f} min"
            )

    if filled < n_samples:

        X_res = X_res[
            :filled
        ]

        y_res = y_res[
            :filled
        ]

    permutation = rng.permutation(
        len(y_res)
    )

    X_res = X_res[
        permutation
    ]

    y_res = y_res[
        permutation
    ]

    tee_print(
        f"[{label}] complete: "
        f"seen={seen:,}, returned={len(y_res):,}, "
        f"elapsed={(time.time()-t0)/60:.2f} min"
    )

    return X_res, y_res


def class_quota_sample_period(
    dates,
    per_class,
    seed,
    label,
):
    rng = np.random.default_rng(seed)

    if per_class <= 0:
        raise ValueError("per_class must be positive.")

    X_parts = [
        []
        for _ in range(N_CLASSES)
    ]

    y_parts = [
        []
        for _ in range(N_CLASSES)
    ]

    counts = np.zeros(
        N_CLASSES,
        dtype=np.int64,
    )

    t0 = time.time()

    for raw in iter_period_chunks(dates):

        X_chunk, y_chunk = raw_to_features(raw)

        for class_id in range(N_CLASSES):

            remaining = (
                per_class
                - counts[class_id]
            )

            if remaining <= 0:
                continue

            indices = np.flatnonzero(
                y_chunk == class_id
            )

            if len(indices) == 0:
                continue

            if len(indices) > remaining:

                chosen = rng.choice(
                    indices,
                    size=int(remaining),
                    replace=False,
                )

            else:

                chosen = indices

            if len(chosen):

                X_parts[class_id].append(
                    X_chunk[
                        chosen
                    ].copy()
                )

                y_parts[class_id].append(
                    y_chunk[
                        chosen
                    ].copy()
                )

                counts[class_id] += len(chosen)

        if np.all(
            counts >= per_class
        ):
            break

        del X_chunk, y_chunk, raw

    missing = np.flatnonzero(
        counts < per_class
    )

    if len(missing):

        raise RuntimeError(
            f"{label}: quota could not be met for classes: "
            + repr(
                [
                    ID_TO_CLASS[int(i)]
                    for i in missing
                ]
            )
        )

    X = np.concatenate(
        [
            np.concatenate(
                X_parts[c],
                axis=0,
            )
            for c in range(N_CLASSES)
        ],
        axis=0,
    )

    y = np.concatenate(
        [
            np.concatenate(
                y_parts[c],
                axis=0,
            )
            for c in range(N_CLASSES)
        ],
        axis=0,
    )

    permutation = rng.permutation(
        len(y)
    )

    X = X[permutation]
    y = y[permutation]

    counts_final = np.bincount(
        y,
        minlength=N_CLASSES,
    )

    assert np.all(
        counts_final == per_class
    )

    tee_print(
        f"[{label}] complete: "
        f"{len(y):,} rows, "
        f"{per_class} per class, "
        f"elapsed={(time.time()-t0)/60:.2f} min"
    )

    del X_parts, y_parts
    gc.collect()

    return X, y


tee_print("Sampling utilities ready.")


In [ ]:
# ============================================================
# Cell 6 — TRAIN-ONLY PREPROCESSING + DATA PREPARATION
# ============================================================

CONTINUOUS_INDICES = np.concatenate(
    [
        np.arange(0, 30),      # IPT
        np.arange(60, 90),     # SIZE
        np.arange(90, 101),    # 11 auxiliary features
    ]
).astype(np.int32)

assert len(CONTINUOUS_INDICES) == 71


class Pass5Preprocessor:
    def __init__(self, chunk_size=PREPROCESS_CHUNK):
        self.chunk_size = int(chunk_size)
        self.scaler = StandardScaler()
        self.fitted = False

    def _prepare(self, X):
        Z = np.asarray(
            X[:, CONTINUOUS_INDICES],
            dtype=np.float32,
        ).copy()
        np.maximum(Z, 0.0, out=Z)
        np.log1p(Z, out=Z)
        return Z

    def partial_fit(self, X):
        Z = self._prepare(X)
        self.scaler.partial_fit(Z)
        del Z
        return self

    def fit_array(self, X):
        n = len(X)
        for start in range(0, n, self.chunk_size):
            stop = min(
                start + self.chunk_size,
                n,
            )
            self.partial_fit(
                X[start:stop]
            )
        self.fitted = True
        return self

    def transform_inplace(self, X):
        if not self.fitted:
            raise RuntimeError(
                "Preprocessor is not fitted."
            )

        n = len(X)

        for start in range(0, n, self.chunk_size):

            stop = min(
                start + self.chunk_size,
                n,
            )

            chunk = X[
                start:stop
            ]

            direction = chunk[:, 30:60].copy()
            padding_mask = (
                direction == 0
            )

            Z = self._prepare(chunk)

            Z = self.scaler.transform(
                Z
            ).astype(
                np.float32,
                copy=False,
            )

            chunk[
                :,
                CONTINUOUS_INDICES,
            ] = Z

            chunk[:, 0:30][padding_mask] = 0.0
            chunk[:, 60:90][padding_mask] = 0.0

            del direction
            del padding_mask
            del Z

        return X


def assert_finite_chunked(
    X,
    chunk_size=PREPROCESS_CHUNK,
):
    n = len(X)

    for start in range(
        0,
        n,
        chunk_size,
    ):

        stop = min(
            start + chunk_size,
            n,
        )

        if not np.isfinite(
            X[start:stop]
        ).all():

            raise ValueError(
                f"Non-finite feature values in "
                f"rows {start}:{stop}."
            )


def verify_direction_values(X):
    values = set()

    for start in range(
        0,
        len(X),
        PREPROCESS_CHUNK,
    ):

        stop = min(
            start + PREPROCESS_CHUNK,
            len(X),
        )

        values.update(
            float(v)
            for v in np.unique(
                X[
                    start:stop,
                    30:60,
                ]
            )
        )

    values = sorted(values)

    assert set(values).issubset(
        {
            -1.0,
            0.0,
            1.0,
        }
    )

    return values


# ------------------------------------------------------------
# Prepare TRAIN
# ------------------------------------------------------------

if DRY_RUN:

    X_train, y_train = (
        class_quota_sample_period(
            TRAIN_DATES,
            DRY_TRAIN_PER_CLASS,
            SEED,
            "DRY-TRAIN",
        )
    )

    TRAIN_ACTUAL_ROWS = len(
        y_train
    )

    train_counts = np.bincount(
        y_train,
        minlength=N_CLASSES,
    ).astype(
        np.int64
    )

    preprocessor = (
        Pass5Preprocessor()
    )

    preprocessor.fit_array(
        X_train
    )

else:

    TRAIN_X_PATH = (
        SCRATCH_DIR / "X_train.npy"
    )

    TRAIN_Y_PATH = (
        SCRATCH_DIR / "y_train.npy"
    )

    tee_print(
        f"Allocating disk-backed X_train: "
        f"{TRAIN_MATRIX_GIB:.2f} GiB"
    )

    X_mm = np.lib.format.open_memmap(
        TRAIN_X_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(
            TRAIN_EXPECTED_ROWS,
            N_FEATURES,
        ),
    )

    y_mm = np.lib.format.open_memmap(
        TRAIN_Y_PATH,
        mode="w+",
        dtype=np.int16,
        shape=(
            TRAIN_EXPECTED_ROWS,
        ),
    )

    write_pos = 0
    t0 = time.time()

    for raw in iter_period_chunks(
        TRAIN_DATES
    ):

        X_chunk, y_chunk = (
            raw_to_features(
                raw
            )
        )

        n = len(y_chunk)
        stop = write_pos + n

        if stop > TRAIN_EXPECTED_ROWS:
            raise RuntimeError(
                "Training row count exceeded expected value."
            )

        X_mm[
            write_pos:stop
        ] = X_chunk

        y_mm[
            write_pos:stop
        ] = y_chunk

        write_pos = stop

        if (
            write_pos
            and
            write_pos // 1_000_000
            !=
            (write_pos - n) // 1_000_000
        ):

            tee_print(
                f"[BUILD] "
                f"{write_pos:,}/{TRAIN_EXPECTED_ROWS:,} "
                f"({100*write_pos/TRAIN_EXPECTED_ROWS:.1f}%) "
                f"elapsed={(time.time()-t0)/60:.2f} min"
            )

        del X_chunk
        del y_chunk
        del raw
        gc.collect()

    X_mm.flush()
    y_mm.flush()

    assert write_pos == TRAIN_EXPECTED_ROWS

    del X_mm
    del y_mm
    gc.collect()

    X_train = np.load(
        TRAIN_X_PATH,
        mmap_mode="r+",
    )

    y_train = np.load(
        TRAIN_Y_PATH,
        mmap_mode="r",
    )

    TRAIN_ACTUAL_ROWS = len(
        y_train
    )

    train_counts = np.bincount(
        y_train,
        minlength=N_CLASSES,
    ).astype(
        np.int64
    )

    assert TRAIN_ACTUAL_ROWS == (
        TRAIN_EXPECTED_ROWS
    )

    assert int(
        train_counts.sum()
    ) == TRAIN_EXPECTED_ROWS

    assert np.all(
        train_counts > 0
    )

    tee_print(
        "\nFitting preprocessing statistics "
        "on COMPLETE training population..."
    )

    preprocessor = (
        Pass5Preprocessor()
    )

    preprocessor.fit_array(
        X_train
    )

# ------------------------------------------------------------
# Prepare VALIDATION
# ------------------------------------------------------------

if DRY_RUN:

    X_val, y_val = (
        class_quota_sample_period(
            VAL_DATES,
            DRY_VAL_PER_CLASS,
            VAL_SEED,
            "DRY-VAL",
        )
    )

else:

    X_val, y_val = (
        reservoir_sample_period(
            VAL_DATES,
            VAL_SAMPLES_FULL,
            VAL_SEED,
            "APRIL-VAL",
        )
    )

# Full training preprocessor is fitted now; transform both sets.
preprocessor.fitted = True

preprocessor.transform_inplace(
    X_train
)

preprocessor.transform_inplace(
    X_val
)

assert_finite_chunked(
    X_train
)

assert_finite_chunked(
    X_val
)

direction_values = (
    verify_direction_values(
        X_train
    )
)

# ------------------------------------------------------------
# Class weights:
# inverse train frequency -> mean normalize -> clip [0.25,5].
# For the full run, this exactly uses the complete train counts.
# ------------------------------------------------------------

class_weights = (
    TRAIN_ACTUAL_ROWS
    / (
        N_CLASSES
        * train_counts.astype(
            np.float64
        )
    )
)

class_weights = (
    class_weights
    / class_weights.mean()
)

class_weights = np.clip(
    class_weights,
    0.25,
    5.0,
).astype(
    np.float32
)

# ------------------------------------------------------------
# Sample weights.
# Full mode uses a disk-backed weight vector.
# ------------------------------------------------------------

if DRY_RUN:

    train_sample_weight = (
        class_weights[
            y_train
        ].astype(
            np.float32,
            copy=False,
        )
    )

    WEIGHT_PATH = (
        SCRATCH_DIR
        / "train_sample_weight.npy"
    )

    np.save(
        WEIGHT_PATH,
        train_sample_weight,
    )

else:

    WEIGHT_PATH = (
        SCRATCH_DIR
        / "train_sample_weight.npy"
    )

    W_mm = np.lib.format.open_memmap(
        WEIGHT_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(
            TRAIN_EXPECTED_ROWS,
        ),
    )

    for start in range(
        0,
        TRAIN_EXPECTED_ROWS,
        1_000_000,
    ):

        stop = min(
            start + 1_000_000,
            TRAIN_EXPECTED_ROWS,
        )

        W_mm[
            start:stop
        ] = class_weights[
            y_train[
                start:stop
            ]
        ]

    W_mm.flush()
    del W_mm

    train_sample_weight = np.load(
        WEIGHT_PATH,
        mmap_mode="r",
    )

# ------------------------------------------------------------
# Save artifacts.
# ------------------------------------------------------------

np.save(
    RUN_DIR / "train_class_counts.npy",
    train_counts,
)

np.save(
    RUN_DIR / "class_weights.npy",
    class_weights,
)

np.save(
    SCRATCH_DIR / "y_val.npy",
    y_val,
)

np.save(
    SCRATCH_DIR / "X_val.npy",
    X_val,
)

joblib.dump(
    preprocessor,
    RUN_DIR / "preprocessor.joblib",
)

if DRY_RUN:
    np.save(
        SCRATCH_DIR / "X_train.npy",
        np.asarray(
            X_train,
            dtype=np.float32,
        ),
    )

# ------------------------------------------------------------
# Final preparation checks.
# ------------------------------------------------------------

assert (
    len(X_train)
    == len(y_train)
)

assert (
    len(X_val)
    == len(y_val)
)

assert X_train.shape[1] == 101
assert X_val.shape[1] == 101

assert np.array_equal(
    np.unique(y_train),
    np.arange(N_CLASSES),
)

assert np.array_equal(
    np.unique(y_val),
    np.arange(N_CLASSES),
)

assert np.isfinite(
    np.asarray(
        train_sample_weight[
            :min(
                len(train_sample_weight),
                100_000,
            )
        ]
    )
).all()

tee_print(
    "\nFULL PREPARATION STAGE COMPLETE",
    flush=True,
)
tee_print(
    "Prepared TRAIN:",
    X_train.shape,
)

tee_print(
    "Prepared VAL  :",
    X_val.shape,
)

tee_print(
    "Training class coverage:",
    np.count_nonzero(
        train_counts
    ),
    "/",
    N_CLASSES,
)

tee_print(
    "Training class count range:",
    int(train_counts.min()),
    "->",
    int(train_counts.max()),
)

tee_print(
    "Direction values:",
    direction_values,
)

tee_print(
    "Class-weight range:",
    float(class_weights.min()),
    "->",
    float(class_weights.max()),
)

tee_print(
    "\nTraining-only preprocessing complete."
)

tee_print(
    "TEST DATA HAS NOT BEEN READ IN FULL MODE."
)


In [ ]:
# ============================================================
# Cell 7 — BASELINE PRE-FLIGHT (BINARY OVR XGBOOST)
# ============================================================

tee_print("=" * 80)
tee_print("BASELINE PRE-FLIGHT")
tee_print("=" * 80)

assert X_train.shape[1] == 101
assert X_val.shape[1] == 101
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert np.array_equal(np.unique(y_train), np.arange(N_CLASSES))
assert np.array_equal(np.unique(y_val), np.arange(N_CLASSES))

probe_rng = np.random.default_rng(SEED + 502)
probe_y = np.tile(np.arange(N_CLASSES, dtype=np.int32), 8)
probe_X = probe_rng.normal(size=(len(probe_y), N_FEATURES)).astype(np.float32)
probe_X[:, 0] = (
    probe_y.astype(np.float32)
    + probe_rng.normal(0.0, 0.05, size=len(probe_y)).astype(np.float32)
)
probe_order = probe_rng.permutation(len(probe_y))
probe_X = probe_X[probe_order]
probe_y = probe_y[probe_order]
probe_w = np.ones(len(probe_y), dtype=np.float32)
probe_y_binary = (probe_y == 0).astype(np.float32)

# Verify LightGBM on the exact selected device.
probe_lgb_kwargs = {
    "objective": "multiclass",
    "num_class": N_CLASSES,
    "n_estimators": 1,
    "num_leaves": 7,
    "min_child_samples": 1,
    "max_bin": 63,
    "random_state": SEED,
    "verbosity": -1,
    "n_jobs": 2,
}
if DRY_RUN:
    probe_lgb_kwargs.update({"device_type": "cpu", "force_col_wise": True})
else:
    probe_lgb_kwargs.update({"device_type": "gpu", "gpu_device_id": 0, "gpu_use_dp": False})
probe_lgb = LGBMClassifier(**probe_lgb_kwargs)
probe_lgb.fit(probe_X, probe_y, sample_weight=probe_w)
del probe_lgb

def free_cupy_pool():
    """Release only unused CuPy blocks; never frees active allocations."""
    if not CUPY_AVAILABLE:
        return
    try:
        with cp.cuda.Device(0):
            cp.cuda.runtime.deviceSynchronize()
        if CUPY_ASYNC_POOL_AVAILABLE and CUPY_ASYNC_POOL is not None:
            CUPY_ASYNC_POOL.free_all_blocks()
        with cp.cuda.Device(0):
            cp.cuda.runtime.deviceSynchronize()
    except Exception as exc:
        tee_print("Note: CuPy pool cleanup unavailable:", repr(exc))

if not DRY_RUN:
    if not GPU_VISIBLE:
        raise RuntimeError("Full run requires a visible NVIDIA GPU.")
    if not CUPY_AVAILABLE:
        raise RuntimeError("Full XGBoost OVR run requires CuPy.")

    # Small CUDA binary-classifier probe.
    gpu_probe = XGBClassifier(
        objective="binary:logistic",
        n_estimators=1,
        max_depth=3,
        learning_rate=0.1,
        tree_method="hist",
        device="cuda",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=2,
        verbosity=0,
    )
    try:
        gpu_probe.fit(probe_X, probe_y_binary, sample_weight=probe_w, verbose=False)
        _ = gpu_probe.predict_proba(probe_X[:16])
    except Exception as exc:
        raise RuntimeError("XGBoost CUDA binary probe failed; stopping before matrix preparation.") from exc
    del gpu_probe
    gc.collect()
    free_cupy_pool()

    # Test the exact data-only ExtMemQuantileDMatrix -> set_label/weight -> train path.
    probe_cache_dir = RUN_DIR / "xgb_external_memory_probe"
    if probe_cache_dir.exists():
        shutil.rmtree(probe_cache_dir)
    probe_cache_dir.mkdir(parents=True, exist_ok=True)

    class ProbeFeatureIter(xgb.DataIter):
        def __init__(self):
            self.X = np.tile(probe_X, (3, 1)).astype(np.float32)
            self.pos = 0
            super().__init__(cache_prefix=str(probe_cache_dir / "probe"), release_data=True, on_host=True)
        def reset(self):
            self.pos = 0
        def next(self, input_data):
            if self.pos >= len(self.X):
                return False
            stop = min(self.pos + 100, len(self.X))
            X_gpu = cp.asarray(self.X[self.pos:stop])
            input_data(data=X_gpu)
            self.pos = stop
            del X_gpu
            return True

    probe_iter = ProbeFeatureIter()
    with xgb.config_context(use_cuda_async_pool=True):
        probe_dmatrix = xgb.ExtMemQuantileDMatrix(
            probe_iter,
            max_bin=FULL_XGB_MAX_BIN,
            cache_host_ratio=XGB_CACHE_HOST_RATIO,
        )
        probe_dmatrix.set_label(np.tile(probe_y_binary, 3))
        probe_dmatrix.set_weight(np.ones(len(probe_y_binary) * 3, dtype=np.float32))
        probe_booster = xgb.train(
            {
                "objective": "binary:logistic",
                "tree_method": "hist",
                "device": "cuda",
                "max_bin": FULL_XGB_MAX_BIN,
                "max_depth": 2,
                "eval_metric": "logloss",
                "verbosity": 0,
                "seed": SEED,
            },
            probe_dmatrix,
            num_boost_round=1,
        )
    del probe_booster, probe_dmatrix, probe_iter
    gc.collect()
    free_cupy_pool()
    shutil.rmtree(probe_cache_dir, ignore_errors=True)

    # Binary logistic needs only grad+hess per row, not per-row-per-class.
    XGB_ESTIMATED_GRAD_HESS_GIB = (
        TRAIN_EXPECTED_ROWS * 2 * 4 / (1024 ** 3)
    )
    with cp.cuda.Device(0):
        cp.cuda.runtime.deviceSynchronize()
        _free_b, _total_b = cp.cuda.runtime.memGetInfo()
    XGB_GPU_FREE_GIB_PREFLIGHT = _free_b / (1024 ** 3)
    XGB_GPU_TOTAL_GIB_PREFLIGHT = _total_b / (1024 ** 3)
    _required_gib = XGB_ESTIMATED_GRAD_HESS_GIB + XGB_GPU_HEADROOM_GIB
    tee_print("XGBoost per-binary gradient/Hessian lower bound:", f"{XGB_ESTIMATED_GRAD_HESS_GIB:.2f} GiB")
    tee_print("CUDA GPU 0 free after probes:", f"{XGB_GPU_FREE_GIB_PREFLIGHT:.2f} / {XGB_GPU_TOTAL_GIB_PREFLIGHT:.2f} GiB")
    if XGB_GPU_FREE_GIB_PREFLIGHT < _required_gib:
        raise RuntimeError(
            f"Insufficient GPU memory for binary XGBoost OVR preflight: "
            f"free={XGB_GPU_FREE_GIB_PREFLIGHT:.2f} GiB, required="
            f"{_required_gib:.2f} GiB. Stopping before any long fit."
        )

    tee_print("LightGBM GPU probe: PASSED")
    tee_print("XGBoost binary external-memory GPU probe: PASSED")
else:
    XGB_ESTIMATED_GRAD_HESS_GIB = (
        len(y_train) * 2 * 4 / (1024 ** 3)
    )
    tee_print("LightGBM CPU probe: PASSED")
    tee_print("XGBoost dry-run uses CPU binary one-vs-rest classifiers.")

del probe_X, probe_y, probe_y_binary, probe_w, probe_order, probe_rng
gc.collect()

tee_print("\nBASELINE PRE-FLIGHT PASSED.")
tee_print("Train rows:", f"{len(y_train):,}")
tee_print("Validation rows:", f"{len(y_val):,}")
tee_print("XGBoost strategy: 50 binary one-vs-rest models; full train population for every class model.")
tee_print("Test data remain untouched in full mode.")


In [ ]:
# ============================================================
# Cell 8 — METRICS + EVALUATION UTILITIES
# ============================================================

def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=10,
):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(
        probabilities,
        dtype=np.float32,
    )

    predictions = probabilities.argmax(
        axis=1
    )

    confidence = probabilities.max(
        axis=1
    )

    correctness = (
        predictions == y_true
    ).astype(
        np.float32
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for b in range(n_bins):

        left = edges[b]
        right = edges[b + 1]

        if b == n_bins - 1:

            mask = (
                confidence >= left
            ) & (
                confidence <= right
            )

        else:

            mask = (
                confidence >= left
            ) & (
                confidence < right
            )

        if not np.any(mask):
            continue

        ece += (
            mask.mean()
            * abs(
                float(
                    correctness[mask].mean()
                )
                -
                float(
                    confidence[mask].mean()
                )
            )
        )

    return float(ece)


def fixed_class_probabilities(
    model,
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float32,
    )

    if probabilities.shape[1] == N_CLASSES:
        return probabilities

    if not hasattr(
        model,
        "classes_",
    ):
        raise ValueError(
            "Probability columns do not cover 50 classes "
            "and the model has no classes_ mapping."
        )

    classes = np.asarray(
        model.classes_,
        dtype=np.int64,
    )

    fixed = np.zeros(
        (
            len(probabilities),
            N_CLASSES,
        ),
        dtype=np.float32,
    )

    fixed[
        :,
        classes,
    ] = probabilities

    return fixed


def metrics_from_predictions(
    y_true,
    predictions,
    probabilities=None,
):
    out = {
        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    predictions,
                )
            ),

        "macro_precision":
            float(
                precision_score(
                    y_true,
                    predictions,
                    labels=np.arange(N_CLASSES),
                    average="macro",
                    zero_division=0,
                )
            ),

        "macro_recall":
            float(
                recall_score(
                    y_true,
                    predictions,
                    labels=np.arange(N_CLASSES),
                    average="macro",
                    zero_division=0,
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    y_true,
                    predictions,
                    labels=np.arange(N_CLASSES),
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    y_true,
                    predictions,
                    average="weighted",
                    zero_division=0,
                )
            ),

        "samples":
            int(len(y_true)),
    }

    out["ece_10"] = (
        expected_calibration_error(
            y_true,
            probabilities,
            10,
        )
        if probabilities is not None
        else float("nan")
    )

    return out


def confusion_matrix_from_predictions(
    y_true,
    predictions,
):
    cm = np.zeros(
        (
            N_CLASSES,
            N_CLASSES,
        ),
        dtype=np.int64,
    )

    np.add.at(
        cm,
        (
            np.asarray(y_true),
            np.asarray(predictions),
        ),
        1,
    )

    return cm


def metrics_from_confusion_matrix(
    cm,
):
    cm = np.asarray(
        cm,
        dtype=np.float64,
    )

    total = cm.sum()

    if total <= 0:
        raise ValueError(
            "Empty confusion matrix."
        )

    tp = np.diag(cm)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)

    precision = np.divide(
        tp,
        predicted,
        out=np.zeros_like(tp),
        where=predicted > 0,
    )

    recall = np.divide(
        tp,
        support,
        out=np.zeros_like(tp),
        where=support > 0,
    )

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    return {
        "accuracy":
            float(
                np.trace(cm)
                / total
            ),

        "macro_precision":
            float(
                precision.mean()
            ),

        "macro_recall":
            float(
                recall.mean()
            ),

        "macro_f1":
            float(
                f1.mean()
            ),

        "weighted_f1":
            float(
                (f1 * support).sum()
                / total
            ),

        "samples":
            int(total),
    }


def save_metrics_bundle(
    model_name,
    split_name,
    y_true,
    predictions,
    probabilities=None,
    extra=None,
):
    metrics = metrics_from_predictions(
        y_true,
        predictions,
        probabilities,
    )

    if extra:
        metrics.update(extra)

    slug = (
        model_name.lower()
        .replace(" ", "_")
    )

    split_slug = (
        split_name.lower()
        .replace(" ", "_")
    )

    cm = confusion_matrix_from_predictions(
        y_true,
        predictions,
    )

    np.save(
        RUN_DIR
        / f"{slug}_{split_slug}_predictions.npy",
        predictions,
    )

    np.save(
        RUN_DIR
        / f"{slug}_{split_slug}_labels.npy",
        y_true,
    )

    np.save(
        RUN_DIR
        / f"{slug}_{split_slug}_confusion_matrix.npy",
        cm,
    )

    report = classification_report(
        y_true,
        predictions,
        labels=np.arange(N_CLASSES),
        target_names=KNOWN_APPS,
        output_dict=True,
        zero_division=0,
    )

    pd.DataFrame(
        report
    ).T.to_csv(
        RUN_DIR
        / f"{slug}_{split_slug}_classification_report.csv"
    )

    with open(
        RUN_DIR
        / f"{slug}_{split_slug}_metrics.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            metrics,
            f,
            indent=2,
        )

    return metrics, cm


tee_print("Metric utilities ready.")


In [ ]:
# ============================================================
# Cell 9 — DISJOINT APRIL PARTITIONS + MATCHED-BUDGET HPO
# ============================================================

from sklearn.linear_model import LogisticRegression

tee_print("=" * 80)
tee_print("APRIL PARTITIONS + BUDGETED HYPERPARAMETER SEARCH")
tee_print("=" * 80)

if DRY_RUN:
    # Diagnostic path only: deliberately do not claim disjoint-stack calibration
    # or paper-valid HPO metrics on the tiny class-quota subset.
    X_hpo_val, y_hpo_val = X_val, y_val
    X_cal_fit = y_cal_fit = X_cal_audit = y_cal_audit = None
    HPO_LGBM_RESULTS = []
    HPO_XGB_RESULTS = []
    LGBM_BEST_PARAMS = {
        "learning_rate": 0.05, "num_leaves": 31,
        "min_child_samples": 20, "max_bin": 63,
        "reg_alpha": 0.0, "reg_lambda": 1.0,
        "subsample": 0.8, "colsample_bytree": 0.8,
    }
    XGB_BEST_PARAMS = {
        "learning_rate": 0.08, "max_depth": 5,
        "min_child_weight": 1, "max_bin": 64,
        "reg_alpha": 0.0, "reg_lambda": 10.0,
        "subsample": 0.8, "colsample_bytree": 0.8,
    }
    LGBM_BEST_ITERATION_HPO = DRY_LGBM_ESTIMATORS
    XGB_HPO_BEST_ROUNDS = [DRY_XGB_ESTIMATORS] * N_CLASSES
    HPO_FAMILY_SUMMARY = {"mode": "dry_run", "paper_valid": False}
    FINAL_LGBM_ROUNDS = DRY_LGBM_ESTIMATORS
    FINAL_XGB_ROUNDS = DRY_XGB_ESTIMATORS
    CALIBRATION_ACTIVE = False
else:
    if len(y_val) != VAL_SAMPLES_FULL:
        raise RuntimeError(f"Expected {VAL_SAMPLES_FULL:,} April reservoir rows, got {len(y_val):,}.")

    # One deterministic permutation creates four mutually disjoint sets.
    # The early-stop set becomes X_val/y_val because the base-model cells use those names.
    _partition_rng = np.random.default_rng(SEED + 104)
    _partition_idx = _partition_rng.permutation(len(y_val))
    X_val_full, y_val_full = X_val, y_val

    _i0 = HPO_VALIDATION_ROWS
    _i1 = _i0 + 70_000
    _i2 = _i1 + 400_000
    _i3 = _i2 + 100_000
    assert _i3 == VAL_SAMPLES_FULL

    X_hpo_val = X_val_full[_partition_idx[:_i0]]
    y_hpo_val = y_val_full[_partition_idx[:_i0]].astype(np.int16, copy=True)
    X_val = X_val_full[_partition_idx[_i0:_i1]]
    y_val = y_val_full[_partition_idx[_i0:_i1]].astype(np.int16, copy=True)
    X_cal_fit = X_val_full[_partition_idx[_i1:_i2]]
    y_cal_fit = y_val_full[_partition_idx[_i1:_i2]].astype(np.int16, copy=True)
    X_cal_audit = X_val_full[_partition_idx[_i2:_i3]]
    y_cal_audit = y_val_full[_partition_idx[_i2:_i3]].astype(np.int16, copy=True)

    assert len(X_hpo_val) == 30_000 and len(X_val) == 70_000
    assert len(X_cal_fit) == 400_000 and len(X_cal_audit) == 100_000
    assert np.array_equal(np.unique(y_val), np.arange(N_CLASSES))
    assert np.array_equal(np.unique(y_cal_fit), np.arange(N_CLASSES))
    assert np.array_equal(np.unique(y_cal_audit), np.arange(N_CLASSES))

    PARTITION_MANIFEST = {
        "seed": SEED + 104,
        "method": "one deterministic permutation of the April uniform reservoir; non-overlapping index ranges",
        "hpo_validation_rows": int(len(y_hpo_val)),
        "early_stopping_rows": int(len(y_val)),
        "stacker_fit_rows": int(len(y_cal_fit)),
        "calibration_audit_rows": int(len(y_cal_audit)),
        "disjoint": True,
    }
    with open(RUN_DIR / "pass5_april_partition_manifest.json", "w", encoding="utf-8") as f:
        json.dump(PARTITION_MANIFEST, f, indent=2)
    np.save(SCRATCH_DIR / "X_val_early.npy", X_val)
    np.save(SCRATCH_DIR / "y_val_early.npy", y_val)

    del X_val_full, y_val_full, _partition_idx, _partition_rng
    gc.collect()

    # Read one common balanced training pilot for both families. Weight correction
    # restores the weighted full-population objective after class-quota sampling.
    X_hpo_pilot, y_hpo_pilot = class_quota_sample_period(
        TRAIN_DATES, HPO_PILOT_PER_CLASS, SEED + 303, "HPO-TRAIN-PILOT"
    )
    assert len(y_hpo_pilot) == HPO_PILOT_ROWS
    pilot_counts = np.bincount(y_hpo_pilot, minlength=N_CLASSES).astype(np.int64)
    assert np.all(pilot_counts == HPO_PILOT_PER_CLASS), pilot_counts
    preprocessor.transform_inplace(X_hpo_pilot)

    pilot_weight = (
        class_weights[y_hpo_pilot].astype(np.float64)
        * train_counts[y_hpo_pilot].astype(np.float64)
        / pilot_counts[y_hpo_pilot].astype(np.float64)
    )
    pilot_weight /= max(float(pilot_weight.mean()), 1e-12)
    pilot_weight = pilot_weight.astype(np.float32)
    assert np.isfinite(pilot_weight).all()

    # Four LightGBM candidate configurations (maximum 240 rounds per HPO trial).
    LGBM_CANDIDATES = [
        {"learning_rate": 0.03, "num_leaves": 31, "min_child_samples": 20, "max_bin": 63, "reg_alpha": 0.0, "reg_lambda": 1.0, "subsample": 0.8, "colsample_bytree": 0.8},
        {"learning_rate": 0.05, "num_leaves": 63, "min_child_samples": 50, "max_bin": 63, "reg_alpha": 0.0, "reg_lambda": 5.0, "subsample": 0.8, "colsample_bytree": 0.9},
        {"learning_rate": 0.05, "num_leaves": 63, "min_child_samples": 100, "max_bin": 31, "reg_alpha": 0.0, "reg_lambda": 1.0, "subsample": 0.9, "colsample_bytree": 0.8},
        {"learning_rate": 0.08, "num_leaves": 31, "min_child_samples": 250, "max_bin": 63, "reg_alpha": 0.1, "reg_lambda": 10.0, "subsample": 0.9, "colsample_bytree": 0.8},
    ]

    HPO_LGBM_RESULTS = []
    _lgb_hpo_start = time.time()
    for _trial, _params in enumerate(LGBM_CANDIDATES, start=1):
        _t0 = time.time()
        _model = LGBMClassifier(
            objective="multiclass", num_class=N_CLASSES,
            n_estimators=HPO_LGBM_MAX_ROUNDS,
            **_params, boosting_type="gbdt", random_state=SEED + _trial,
            n_jobs=4, verbosity=-1, device_type="gpu", gpu_device_id=0,
            gpu_use_dp=False, subsample_freq=1,
        )
        _model.fit(
            X_hpo_pilot, y_hpo_pilot, sample_weight=pilot_weight,
            eval_set=[(X_hpo_val, y_hpo_val)], eval_metric="multi_logloss",
            callbacks=[
                lgb.early_stopping(HPO_LGBM_EARLY_STOPPING, first_metric_only=True, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )
        _best_iter = int(getattr(_model, "best_iteration_", HPO_LGBM_MAX_ROUNDS) or HPO_LGBM_MAX_ROUNDS)
        _p = fixed_class_probabilities(_model, _model.predict_proba(X_hpo_val, num_iteration=_best_iter))
        _score = float(f1_score(y_hpo_val, _p.argmax(axis=1), labels=np.arange(N_CLASSES), average="macro", zero_division=0))
        _elapsed = time.time() - _t0
        HPO_LGBM_RESULTS.append({
            "family": "LightGBM", "trial": _trial, "macro_f1": _score,
            "best_iteration": _best_iter, "round_cap_hit": bool(_best_iter >= HPO_LGBM_MAX_ROUNDS),
            "seconds": _elapsed, "params": dict(_params),
        })
        pd.DataFrame([{**r, "params_json": json.dumps(r["params"], sort_keys=True)} for r in HPO_LGBM_RESULTS]).drop(columns=["params"]).to_csv(RUN_DIR / "pass5_hpo_lightgbm_partial.csv", index=False)
        tee_print(f"[LightGBM HPO { _trial }/{len(LGBM_CANDIDATES)}] macro-F1={_score:.5f}; best_iteration={_best_iter}; seconds={_elapsed:.1f}; params={_params}")
        del _p, _model
        gc.collect()
        if time.time() - _lgb_hpo_start >= HPO_BUDGET_SECONDS_PER_FAMILY:
            break

    if not HPO_LGBM_RESULTS:
        raise RuntimeError("No completed LightGBM HPO trial.")
    _lgb_best_record = max(HPO_LGBM_RESULTS, key=lambda r: r["macro_f1"])
    LGBM_BEST_PARAMS = dict(_lgb_best_record["params"])
    LGBM_BEST_ITERATION_HPO = int(_lgb_best_record["best_iteration"])
    LGBM_HPO_SECONDS = float(time.time() - _lgb_hpo_start)

    # Six XGBoost OvR configurations; the loop respects the same 900-second family budget.
    XGB_CANDIDATES = [
        {"learning_rate": 0.05, "max_depth": 4, "min_child_weight": 1, "max_bin": 32, "reg_alpha": 0.0, "reg_lambda": 5.0, "subsample": 0.8, "colsample_bytree": 0.8},
        {"learning_rate": 0.08, "max_depth": 5, "min_child_weight": 1, "max_bin": 64, "reg_alpha": 0.0, "reg_lambda": 10.0, "subsample": 0.8, "colsample_bytree": 0.8},
        {"learning_rate": 0.10, "max_depth": 5, "min_child_weight": 10, "max_bin": 64, "reg_alpha": 0.1, "reg_lambda": 1.0, "subsample": 0.9, "colsample_bytree": 0.8},
        {"learning_rate": 0.15, "max_depth": 5, "min_child_weight": 1, "max_bin": 64, "reg_alpha": 0.0, "reg_lambda": 20.0, "subsample": 0.9, "colsample_bytree": 0.7},
        {"learning_rate": 0.08, "max_depth": 6, "min_child_weight": 5, "max_bin": 32, "reg_alpha": 0.0, "reg_lambda": 5.0, "subsample": 0.8, "colsample_bytree": 0.9},
        {"learning_rate": 0.10, "max_depth": 4, "min_child_weight": 5, "max_bin": 64, "reg_alpha": 0.1, "reg_lambda": 10.0, "subsample": 0.8, "colsample_bytree": 0.9},
    ]

    HPO_XGB_RESULTS = []
    _xgb_hpo_start = time.time()
    for _trial, _params in enumerate(XGB_CANDIDATES, start=1):
        _t0 = time.time()
        _trial_probs = np.empty((len(y_hpo_val), N_CLASSES), dtype=np.float32)
        _rounds = []
        for _class_id in range(N_CLASSES):
            _y_pilot_binary = (y_hpo_pilot == _class_id).astype(np.int8)
            _y_val_binary = (y_hpo_val == _class_id).astype(np.int8)
            _clf = XGBClassifier(
                objective="binary:logistic", n_estimators=HPO_XGB_MAX_ROUNDS,
                learning_rate=_params["learning_rate"], max_depth=_params["max_depth"],
                min_child_weight=_params["min_child_weight"], subsample=_params["subsample"],
                colsample_bytree=_params["colsample_bytree"], reg_alpha=_params["reg_alpha"],
                reg_lambda=_params["reg_lambda"], max_bin=_params["max_bin"],
                tree_method="hist", device="cuda", eval_metric="logloss",
                early_stopping_rounds=HPO_XGB_EARLY_STOPPING,
                random_state=SEED + 5000 * _trial + _class_id, n_jobs=2, verbosity=0,
            )
            _clf.fit(
                X_hpo_pilot, _y_pilot_binary, sample_weight=pilot_weight,
                eval_set=[(X_hpo_val, _y_val_binary)], verbose=False,
            )
            _trial_probs[:, _class_id] = _clf.predict_proba(X_hpo_val)[:, 1].astype(np.float32, copy=False)
            _rounds.append(int(getattr(_clf, "best_iteration", HPO_XGB_MAX_ROUNDS - 1)) + 1)
            del _clf, _y_pilot_binary, _y_val_binary
        _row_sum = _trial_probs.sum(axis=1, keepdims=True)
        _trial_probs /= np.maximum(_row_sum, 1e-12)
        _zero_rows = _row_sum[:, 0] <= 1e-12
        _trial_probs[_zero_rows, :] = 1.0 / N_CLASSES
        _score = float(f1_score(y_hpo_val, _trial_probs.argmax(axis=1), labels=np.arange(N_CLASSES), average="macro", zero_division=0))
        _elapsed = time.time() - _t0
        HPO_XGB_RESULTS.append({
            "family": "XGBoost OvR", "trial": _trial, "macro_f1": _score,
            "median_best_rounds": int(np.median(_rounds)),
            "q75_best_rounds": float(np.quantile(_rounds, 0.75)),
            "cap_hit_fraction": float(np.mean(np.asarray(_rounds) >= HPO_XGB_MAX_ROUNDS)),
            "seconds": _elapsed, "params": dict(_params),
        })
        pd.DataFrame([{**r, "params_json": json.dumps(r["params"], sort_keys=True)} for r in HPO_XGB_RESULTS]).drop(columns=["params"]).to_csv(RUN_DIR / "pass5_hpo_xgboost_partial.csv", index=False)
        tee_print(f"[XGBoost-OvR HPO {_trial}/{len(XGB_CANDIDATES)}] macro-F1={_score:.5f}; median_best_rounds={int(np.median(_rounds))}; cap_hit_fraction={np.mean(np.asarray(_rounds) >= HPO_XGB_MAX_ROUNDS):.2f}; seconds={_elapsed:.1f}; params={_params}")
        del _trial_probs, _row_sum, _zero_rows, _rounds
        gc.collect()
        if time.time() - _xgb_hpo_start >= HPO_BUDGET_SECONDS_PER_FAMILY:
            break

    if not HPO_XGB_RESULTS:
        raise RuntimeError("No completed XGBoost HPO trial.")
    _xgb_best_record = max(HPO_XGB_RESULTS, key=lambda r: r["macro_f1"])
    XGB_BEST_PARAMS = dict(_xgb_best_record["params"])
    XGB_HPO_BEST_ROUNDS = [int(r["median_best_rounds"]) for r in HPO_XGB_RESULTS]
    XGB_HPO_SECONDS = float(time.time() - _xgb_hpo_start)

    # Allow more final rounds than HPO; early stopping is still validation-driven.
    FINAL_LGBM_ROUNDS = min(
        FULL_LGBM_MAX_ROUNDS,
        max(260, int(np.ceil(1.15 * LGBM_BEST_ITERATION_HPO))),
    )
    _selected_xgb_q75 = float(_xgb_best_record["q75_best_rounds"])
    FINAL_XGB_ROUNDS = min(
        FULL_XGB_MAX_ROUNDS,
        max(190, int(np.ceil(1.50 * _selected_xgb_q75))),
    )
    CALIBRATION_ACTIVE = True

    HPO_FAMILY_SUMMARY = {
        "target_wallclock_seconds_per_family": HPO_BUDGET_SECONDS_PER_FAMILY,
        "pilot_rows": int(len(y_hpo_pilot)),
        "pilot_policy": "2,000 per class; inverse sampling correction times the common train-derived per-flow class weight; normalized to mean one",
        "hpo_validation_rows": int(len(y_hpo_val)),
        "selection_metric": "macro-F1 on the same April HPO validation partition",
        "LightGBM": {
            "completed_trials": len(HPO_LGBM_RESULTS), "elapsed_seconds": LGBM_HPO_SECONDS,
            "best_macro_f1": float(_lgb_best_record["macro_f1"]),
            "best_iteration_on_pilot": LGBM_BEST_ITERATION_HPO,
            "selected_params": LGBM_BEST_PARAMS,
            "final_round_cap": FINAL_LGBM_ROUNDS,
            "trial_results": HPO_LGBM_RESULTS,
        },
        "XGBoost_OvR": {
            "completed_trials": len(HPO_XGB_RESULTS), "elapsed_seconds": XGB_HPO_SECONDS,
            "best_macro_f1": float(_xgb_best_record["macro_f1"]),
            "selected_median_best_rounds": int(_xgb_best_record["median_best_rounds"]),
            "selected_q75_best_rounds": _selected_xgb_q75,
            "selected_params": XGB_BEST_PARAMS,
            "final_round_cap_per_binary_model": FINAL_XGB_ROUNDS,
            "trial_results": HPO_XGB_RESULTS,
        },
    }
    with open(RUN_DIR / "pass5_hpo_summary.json", "w", encoding="utf-8") as f:
        json.dump(HPO_FAMILY_SUMMARY, f, indent=2)
    _hpo_rows = []
    for _row in HPO_LGBM_RESULTS + HPO_XGB_RESULTS:
        _copy = dict(_row)
        _copy["params_json"] = json.dumps(_copy.pop("params"), sort_keys=True)
        _hpo_rows.append(_copy)
    pd.DataFrame(_hpo_rows).to_csv(RUN_DIR / "pass5_hpo_trials.csv", index=False)
    tee_print("HPO complete. Family summaries:", json.dumps({k: {kk: vv for kk, vv in v.items() if kk not in ("trial_results", "selected_params")} if isinstance(v, dict) else v for k, v in HPO_FAMILY_SUMMARY.items() if k in ("LightGBM", "XGBoost_OvR")}, indent=2))
    tee_print("Disjoint April partitions:", json.dumps(PARTITION_MANIFEST, indent=2))

# Guarded cleanup: no undefined temporary names are deleted.
for _tmp_name in (
    "X_hpo_pilot", "y_hpo_pilot", "pilot_counts", "pilot_weight",
    "_trial_probs", "_row_sum", "_zero_rows", "_rounds",
    "_y_pilot_binary", "_y_val_binary", "_clf", "_model", "_p",
):
    globals().pop(_tmp_name, None)
gc.collect()

tee_print("Selected LightGBM parameters:", LGBM_BEST_PARAMS)
tee_print("Selected XGBoost OvR parameters:", XGB_BEST_PARAMS)
tee_print("Final iteration caps — LightGBM:", FINAL_LGBM_ROUNDS, "; XGBoost per class:", FINAL_XGB_ROUNDS)
tee_print("April HPO validation data are disjoint from final early stopping and stacker/audit partitions.")
tee_print("May/test data have NOT been read.")

In [ ]:
# ============================================================
# Cell 9 — XGBOOST ONE-VS-REST BASELINE (VRAM-BOUNDED)
# ============================================================

XGB_N_ESTIMATORS = DRY_XGB_ESTIMATORS if DRY_RUN else FINAL_XGB_ROUNDS
XGB_EARLY_STOPPING = DRY_EARLY_STOPPING if DRY_RUN else FULL_XGB_EARLY_STOPPING

XGB_CONFIG = {
    "strategy": "one-vs-rest",
    "objective": "binary:logistic per class",
    "n_models": N_CLASSES,
    "n_estimators_per_class": XGB_N_ESTIMATORS,
    **dict(XGB_BEST_PARAMS),
    "tree_method": "hist",
    "device": "cpu" if DRY_RUN else "cuda",
    "eval_metric": "logloss",
    "random_state": SEED,
    "n_jobs": min(4, os.cpu_count() or 2),
    "verbosity": 0,
    "probability_coupling": "row-normalize independent per-class sigmoid probabilities",
    "hpo_family_budget_seconds": HPO_BUDGET_SECONDS_PER_FAMILY,
    "hpo_trials_completed": len(HPO_XGB_RESULTS),
    "hpo_best_macro_f1": max((r["macro_f1"] for r in HPO_XGB_RESULTS), default=None),
}

tee_print("=" * 80)
tee_print("XGBOOST ONE-VS-REST BASELINE")
tee_print("=" * 80)
tee_print("Training", N_CLASSES, "binary boosters sequentially.")
tee_print("Every class model uses the same full training population:", f"{len(y_train):,}", "rows.")
tee_print(json.dumps({"config": XGB_CONFIG, "early_stopping_rounds": XGB_EARLY_STOPPING,
                      "training_rows_per_model": int(len(y_train)),
                      "validation_rows_per_model": int(len(y_val))}, indent=2))

XGB_EXTERNAL_MEMORY = not DRY_RUN
XGB_OVR_MODEL_DIR = RUN_DIR / "xgboost_ovr_models"
XGB_OVR_MODEL_DIR.mkdir(parents=True, exist_ok=True)
XGB_PROGRESS_PATH = RUN_DIR / "xgboost_ovr_progress.json"
XGB_CACHE_DIR = SCRATCH_DIR / "xgb_cache"

# When rerunning only this cell after an interruption, drop stale DMatrix/iterator
# references before removing the old external-memory cache and reclaim free blocks.
for _stale_name in ("dtrain", "dvalid", "train_iter", "valid_iter"):
    if _stale_name in globals():
        globals().pop(_stale_name, None)
gc.collect()
if not DRY_RUN:
    free_cupy_pool()

# A stable signature allows rerunning this cell after an interrupted OVR loop
# in the same live session: completed class models are reused only if settings match.
XGB_OVR_SIGNATURE_OBJECT = {
    "notebook_protocol": "pass5_fair_comparison_v10.1",
    "preprocessing": "log1p+training-only StandardScaler on 71 continuous features; direction mask restores PPI padding to zero",
    "config": XGB_CONFIG,
    "early_stopping_rounds": XGB_EARLY_STOPPING,
    "training_rows": int(len(y_train)),
    "validation_rows": int(len(y_val)),
    "classes": list(KNOWN_APPS),
    "weighting": "original-class inverse-frequency sample weights clipped [0.25,5.0]",
}
XGB_OVR_SIGNATURE = json.dumps(XGB_OVR_SIGNATURE_OBJECT, sort_keys=True)

XGB_OVR_MODELS = [None] * N_CLASSES
XGB_BEST_ITERATIONS = [None] * N_CLASSES
XGB_CLASS_TRAIN_SECONDS = [0.0] * N_CLASSES
XGB_COMPLETED_CLASSES = set()

if XGB_CACHE_DIR.exists():
    shutil.rmtree(XGB_CACHE_DIR)
XGB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

_progress = {"signature": XGB_OVR_SIGNATURE, "completed": {}, "total_training_seconds": 0.0}
if XGB_PROGRESS_PATH.exists():
    try:
        with open(XGB_PROGRESS_PATH, "r", encoding="utf-8") as f:
            _old_progress = json.load(f)
        if _old_progress.get("signature") == XGB_OVR_SIGNATURE:
            _progress = _old_progress
            for _key, _rec in _progress.get("completed", {}).items():
                _cid = int(_key)
                _model_path = RUN_DIR / _rec["model_file"]
                if 0 <= _cid < N_CLASSES and _model_path.exists():
                    _booster = xgb.Booster()
                    _booster.load_model(str(_model_path))
                    _booster.set_param({"device": XGB_CONFIG["device"], "nthread": min(4, os.cpu_count() or 2)})
                    XGB_OVR_MODELS[_cid] = _booster
                    XGB_BEST_ITERATIONS[_cid] = int(_rec["best_iteration"])
                    XGB_CLASS_TRAIN_SECONDS[_cid] = float(_rec.get("training_seconds", 0.0))
                    XGB_COMPLETED_CLASSES.add(_cid)
            tee_print("Resume check: reusing", len(XGB_COMPLETED_CLASSES), "completed class models with matching configuration.")
    except Exception as _resume_exc:
        tee_print("Resume metadata could not be used; training classes from scratch:", repr(_resume_exc))

# Feature-only iterator: quantized external-memory pages are built once.
# Labels are attached/replaced with DMatrix.set_label for each binary objective.
class FeatureOnlyDataIter(xgb.DataIter):
    def __init__(self, X_source, batch_rows, cache_prefix):
        if isinstance(X_source, (str, Path)):
            self.X = np.load(X_source, mmap_mode="r")
            self._owns_X = True
        else:
            self.X = X_source
            self._owns_X = False
        self.batch_rows = int(batch_rows)
        self.pos = 0
        self.n = int(len(self.X))
        super().__init__(cache_prefix=str(cache_prefix), release_data=True, on_host=True)
    def reset(self):
        self.pos = 0
    def next(self, input_data):
        if self.pos >= self.n:
            return False
        stop = min(self.pos + self.batch_rows, self.n)
        X_chunk = np.asarray(self.X[self.pos:stop], dtype=np.float32)
        X_gpu = cp.asarray(X_chunk)
        input_data(data=X_gpu)
        self.pos = stop
        del X_chunk, X_gpu
        return True

# Prediction helper shared by validation and May-test evaluation.
# A single GPU feature batch is reused across all 50 binary boosters.
def xgb_ovr_predict_batch(models, X_batch, best_iterations, use_cuda):
    X_batch = np.asarray(X_batch, dtype=np.float32)
    n_rows = len(X_batch)
    if use_cuda:
        X_backend = cp.asarray(X_batch)
        probabilities_backend = cp.empty((n_rows, N_CLASSES), dtype=cp.float32)
        for class_id, booster in enumerate(models):
            p_k = booster.inplace_predict(
                X_backend,
                iteration_range=(0, int(best_iterations[class_id]) + 1),
            )
            if not isinstance(p_k, cp.ndarray):
                p_k = cp.asarray(p_k, dtype=cp.float32)
            probabilities_backend[:, class_id] = p_k.astype(cp.float32, copy=False)
        denominator = cp.sum(probabilities_backend, axis=1, keepdims=True)
        zero_rows = cp.squeeze(denominator <= 1e-12, axis=1)
        probabilities_backend /= cp.maximum(denominator, 1e-12)
        if bool(cp.any(zero_rows).item()):
            probabilities_backend[zero_rows, :] = 1.0 / N_CLASSES
        probabilities = cp.asnumpy(probabilities_backend)
        del X_backend, probabilities_backend, denominator, zero_rows
        return probabilities.astype(np.float32, copy=False)
    probabilities = np.empty((n_rows, N_CLASSES), dtype=np.float32)
    for class_id, booster in enumerate(models):
        p_k = booster.inplace_predict(
            X_batch,
            iteration_range=(0, int(best_iterations[class_id]) + 1),
        )
        probabilities[:, class_id] = np.asarray(p_k, dtype=np.float32).reshape(-1)
    denominator = probabilities.sum(axis=1, keepdims=True)
    zero_rows = denominator[:, 0] <= 1e-12
    probabilities /= np.maximum(denominator, 1e-12)
    probabilities[zero_rows, :] = 1.0 / N_CLASSES
    return probabilities

XGB_MATRIX_SECONDS = 0.0
XGB_TRAIN_SECONDS = float(sum(XGB_CLASS_TRAIN_SECONDS))

if not DRY_RUN:
    assert GPU_VISIBLE and CUPY_AVAILABLE
    train_iter = FeatureOnlyDataIter(
        TRAIN_X_PATH, XGB_ITERATOR_ROWS, XGB_CACHE_DIR / "train"
    )
    valid_iter = FeatureOnlyDataIter(
        X_val, XGB_ITERATOR_ROWS, XGB_CACHE_DIR / "valid"
    )
    tee_print("Constructing host-cached external-memory quantile matrices for features only...")
    t0_matrix = time.time()
    with xgb.config_context(use_cuda_async_pool=True):
        dtrain = xgb.ExtMemQuantileDMatrix(
            train_iter, max_bin=XGB_CONFIG["max_bin"], cache_host_ratio=XGB_CACHE_HOST_RATIO
        )
        dvalid = xgb.ExtMemQuantileDMatrix(
            valid_iter, max_bin=XGB_CONFIG["max_bin"], ref=dtrain,
            cache_host_ratio=XGB_CACHE_HOST_RATIO
        )
    XGB_MATRIX_SECONDS = time.time() - t0_matrix
    dtrain.set_weight(np.asarray(train_sample_weight, dtype=np.float32))
    free_cupy_pool()
    with cp.cuda.Device(0):
        cp.cuda.runtime.deviceSynchronize()
        _gpu_free_b, _gpu_total_b = cp.cuda.runtime.memGetInfo()
    XGB_GPU_FREE_GIB_PRE_TRAIN = _gpu_free_b / (1024 ** 3)
    tee_print("Feature matrix construction:", f"{XGB_MATRIX_SECONDS/60:.2f} min")
    tee_print("CUDA free before binary OVR fits:", f"{XGB_GPU_FREE_GIB_PRE_TRAIN:.2f} GiB")
    if XGB_GPU_FREE_GIB_PRE_TRAIN < XGB_ESTIMATED_GRAD_HESS_GIB + XGB_GPU_HEADROOM_GIB:
        del dtrain, dvalid, train_iter, valid_iter
        gc.collect()
        raise RuntimeError("Insufficient GPU headroom for binary OVR XGBoost after matrix construction; stopped before fitting any full-data classifier.")
else:
    dtrain = dvalid = train_iter = valid_iter = None

for class_id in range(N_CLASSES):
    if class_id in XGB_COMPLETED_CLASSES:
        tee_print(f"[XGB OVR] class {class_id+1}/{N_CLASSES} already complete; reusing saved booster.")
        continue

    class_name = KNOWN_APPS[class_id]
    y_binary_train = (np.asarray(y_train) == class_id).astype(np.float32)
    y_binary_val = (np.asarray(y_val) == class_id).astype(np.float32)
    class_model_path = XGB_OVR_MODEL_DIR / f"class_{class_id:02d}.json"
    t0_class = time.time()
    tee_print(f"\n[XGB OVR] class {class_id+1:02d}/{N_CLASSES}: {class_name}; positives={int(y_binary_train.sum()):,}")

    if DRY_RUN:
        classifier = XGBClassifier(
            objective="binary:logistic",
            n_estimators=XGB_N_ESTIMATORS,
            learning_rate=XGB_CONFIG["learning_rate"],
            max_depth=XGB_CONFIG["max_depth"],
            min_child_weight=XGB_CONFIG["min_child_weight"],
            subsample=XGB_CONFIG["subsample"],
            colsample_bytree=XGB_CONFIG["colsample_bytree"],
            reg_alpha=XGB_CONFIG["reg_alpha"],
            reg_lambda=XGB_CONFIG["reg_lambda"],
            max_bin=XGB_CONFIG["max_bin"],
            tree_method="hist",
            device="cpu",
            eval_metric="logloss",
            random_state=SEED + class_id,
            n_jobs=min(4, os.cpu_count() or 2),
            verbosity=0,
            early_stopping_rounds=XGB_EARLY_STOPPING,
        )
        classifier.fit(
            X_train, y_binary_train,
            sample_weight=train_sample_weight,
            eval_set=[(X_val, y_binary_val)],
            verbose=False,
        )
        booster = classifier.get_booster()
        best_iteration = int(getattr(classifier, "best_iteration", XGB_N_ESTIMATORS - 1))
        booster.save_model(str(class_model_path))
        del classifier
    else:
        dtrain.set_label(y_binary_train)
        dvalid.set_label(y_binary_val)
        class_checkpoint_dir = SCRATCH_DIR / "xgb_checkpoints" / f"class_{class_id:02d}"
        # The checkpoint callback is diagnostic/protective storage, not automatic
        # resume state. Clear a stale directory before restarting an incomplete class.
        if class_checkpoint_dir.exists():
            shutil.rmtree(class_checkpoint_dir)
        class_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        params = {
            "objective": "binary:logistic",
            "eta": XGB_CONFIG["learning_rate"],
            "max_depth": XGB_CONFIG["max_depth"],
            "min_child_weight": XGB_CONFIG["min_child_weight"],
            "subsample": XGB_CONFIG["subsample"],
            "colsample_bytree": XGB_CONFIG["colsample_bytree"],
            "alpha": XGB_CONFIG["reg_alpha"],
            "lambda": XGB_CONFIG["reg_lambda"],
            "max_bin": XGB_CONFIG["max_bin"],
            "tree_method": "hist",
            "device": "cuda",
            "eval_metric": "logloss",
            "seed": SEED + class_id,
            "nthread": min(4, os.cpu_count() or 2),
            "verbosity": 0,
        }
        checkpoint_callback = xgb.callback.TrainingCheckPoint(
            directory=str(class_checkpoint_dir),
            name=f"xgb_ovr_{class_id:02d}",
            as_pickle=False,
            interval=XGB_CHECKPOINT_INTERVAL,
        )
        with xgb.config_context(use_cuda_async_pool=True):
            booster = xgb.train(
                params,
                dtrain,
                num_boost_round=XGB_N_ESTIMATORS,
                evals=[(dvalid, "validation")],
                early_stopping_rounds=XGB_EARLY_STOPPING,
                callbacks=[checkpoint_callback],
                verbose_eval=25,
            )
        best_iteration = int(getattr(booster, "best_iteration", XGB_N_ESTIMATORS - 1))
        booster.save_model(str(class_model_path))

    class_seconds = time.time() - t0_class
    XGB_OVR_MODELS[class_id] = booster
    XGB_BEST_ITERATIONS[class_id] = best_iteration
    XGB_CLASS_TRAIN_SECONDS[class_id] = class_seconds
    XGB_COMPLETED_CLASSES.add(class_id)
    XGB_TRAIN_SECONDS = float(sum(XGB_CLASS_TRAIN_SECONDS))

    _progress.setdefault("completed", {})[str(class_id)] = {
        "model_file": str(class_model_path.relative_to(RUN_DIR)),
        "class_name": class_name,
        "best_iteration": best_iteration,
        "training_seconds": class_seconds,
    }
    _progress["total_training_seconds"] = XGB_TRAIN_SECONDS
    _tmp_progress = XGB_PROGRESS_PATH.with_suffix(".tmp")
    with open(_tmp_progress, "w", encoding="utf-8") as f:
        json.dump(_progress, f, indent=2)
    os.replace(_tmp_progress, XGB_PROGRESS_PATH)
    tee_print(f"[XGB OVR] finished {class_name}: best_iteration={best_iteration}, seconds={class_seconds:.1f}; completed={len(XGB_COMPLETED_CLASSES)}/{N_CLASSES}")
    del y_binary_train, y_binary_val
    gc.collect()

assert all(model is not None for model in XGB_OVR_MODELS), "Not all 50 XGBoost OVR models were trained/loaded."
assert all(value is not None for value in XGB_BEST_ITERATIONS)
XGB_BEST_ITERATION = int(round(float(np.median(XGB_BEST_ITERATIONS))))

# Release external-memory matrices before computing probabilities.
if not DRY_RUN:
    del dtrain, dvalid, train_iter, valid_iter
    gc.collect()
    free_cupy_pool()
    if XGB_CACHE_DIR.exists():
        shutil.rmtree(XGB_CACHE_DIR, ignore_errors=True)

tee_print("\nComputing April validation probabilities across all one-vs-rest models...")
t0_val_pred = time.time()
xgb_val_probs = np.empty((len(y_val), N_CLASSES), dtype=np.float32)
for start in range(0, len(y_val), 16384):
    stop = min(start + 16384, len(y_val))
    xgb_val_probs[start:stop] = xgb_ovr_predict_batch(
        XGB_OVR_MODELS, X_val[start:stop], XGB_BEST_ITERATIONS, use_cuda=(not DRY_RUN)
    )
XGB_VAL_PREDICTION_SECONDS = time.time() - t0_val_pred

XGB_VAL_METRICS, XGB_VAL_CM = save_metrics_bundle(
    "XGBoost", "validation", y_val, xgb_val_probs.argmax(axis=1), xgb_val_probs,
    {
        "strategy": "one-vs-rest binary ensemble",
        "best_iteration_median": XGB_BEST_ITERATION,
        "best_iteration_by_class": {KNOWN_APPS[i]: int(XGB_BEST_ITERATIONS[i]) for i in range(N_CLASSES)},
        "training_seconds": XGB_TRAIN_SECONDS,
        "matrix_construction_seconds": XGB_MATRIX_SECONDS,
        "validation_prediction_seconds": XGB_VAL_PREDICTION_SECONDS,
        "train_rows": int(len(y_train)),
        "external_memory": XGB_EXTERNAL_MEMORY,
    },
)

# Canonical bundle metadata: all actual fitted XGBoost models are separate JSON files.
XGB_MODEL_FILES = [
    {"class_id": i, "class_name": KNOWN_APPS[i],
     "file": str((XGB_OVR_MODEL_DIR / f"class_{i:02d}.json").relative_to(RUN_DIR))}
    for i in range(N_CLASSES)
]
XGB_BUNDLE_METADATA = {
    "model_type": "xgboost_one_vs_rest_binary_ensemble",
    "strategy": "one-vs-rest",
    "num_classes": N_CLASSES,
    "class_names": list(KNOWN_APPS),
    "model_files": XGB_MODEL_FILES,
    "best_iteration_by_class": [int(v) for v in XGB_BEST_ITERATIONS],
    "best_iteration_median": XGB_BEST_ITERATION,
    "probability_coupling": "row-normalization of independent binary:logistic positive-class probabilities",
    "training_rows": int(len(y_train)),
    "validation_rows": int(len(y_val)),
    "config": XGB_CONFIG,
}
with open(RUN_DIR / "xgboost_baseline.json", "w", encoding="utf-8") as f:
    json.dump(XGB_BUNDLE_METADATA, f, indent=2)
with open(RUN_DIR / "xgboost_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "config": XGB_CONFIG,
        "early_stopping_rounds": XGB_EARLY_STOPPING,
        "best_iteration": XGB_BEST_ITERATION,
        "best_iteration_by_class": {KNOWN_APPS[i]: int(XGB_BEST_ITERATIONS[i]) for i in range(N_CLASSES)},
        "training_seconds": XGB_TRAIN_SECONDS,
        "matrix_construction_seconds": XGB_MATRIX_SECONDS,
        "validation_prediction_seconds": XGB_VAL_PREDICTION_SECONDS,
        "train_rows": int(len(y_train)),
        "validation_rows": int(len(y_val)),
        "feature_count": N_FEATURES,
        "external_memory": XGB_EXTERNAL_MEMORY,
        "external_memory_settings": {
            "cache_host_ratio": XGB_CACHE_HOST_RATIO if XGB_EXTERNAL_MEMORY else None,
            "iterator_batch_rows": XGB_ITERATOR_ROWS if XGB_EXTERNAL_MEMORY else None,
            "checkpoint_interval_rounds": XGB_CHECKPOINT_INTERVAL if XGB_EXTERNAL_MEMORY else None,
            "estimated_binary_gradient_hessian_gib": float(XGB_ESTIMATED_GRAD_HESS_GIB),
        },
        "device": XGB_CONFIG["device"],
    }, f, indent=2)

# This protocol marker is still written by the following LightGBM cell only after
# the two baseline configurations are frozen; May remains untouched at this point.
tee_print("\nXGBoost OVR baseline frozen (first stage).")
tee_print("Strategy: 50 binary:logistic boosters; probabilities row-normalized.")
tee_print("Median best iteration:", XGB_BEST_ITERATION)
tee_print("Training seconds across class models:", f"{XGB_TRAIN_SECONDS:.1f}")
tee_print("TEST DATA HAS NOT BEEN READ.")


In [ ]:
# ============================================================
# Cell 10 — LIGHTGBM BASELINE (AFTER XGBOOST)
# ============================================================

LGBM_CONFIG = {
    "objective": "multiclass",
    "num_class": N_CLASSES,
    "n_estimators": DRY_LGBM_ESTIMATORS if DRY_RUN else FINAL_LGBM_ROUNDS,
    **dict(LGBM_BEST_PARAMS),
    "max_depth": -1,
    "subsample_freq": 1,
    "boosting_type": "gbdt",
    "random_state": SEED,
    "n_jobs": 4,
    "verbosity": -1,
}

if DRY_RUN:
    LGBM_CONFIG.update({"device_type": "cpu", "force_col_wise": True})
else:
    LGBM_CONFIG.update({"device_type": "gpu", "gpu_device_id": 0, "gpu_use_dp": True})

LGBM_EARLY_STOPPING = DRY_EARLY_STOPPING if DRY_RUN else FULL_LGBM_EARLY_STOPPING

tee_print("=" * 80)
tee_print("LIGHTGBM BASELINE")
tee_print("=" * 80)

tee_print(
    "Starting LightGBM fit...",
    "device=" + LGBM_CONFIG.get("device_type", "cpu"),
    "rows=" + f"{len(y_train):,}",
    flush=True,
)
tee_print(
    json.dumps(
        {
            "config": LGBM_CONFIG,
            "hpo_budget_seconds_per_family": HPO_BUDGET_SECONDS_PER_FAMILY,
            "hpo_trials_completed": len(HPO_LGBM_RESULTS),
            "hpo_best_macro_f1": max((r["macro_f1"] for r in HPO_LGBM_RESULTS), default=None),
            "hpo_best_iteration_on_pilot": LGBM_BEST_ITERATION_HPO,
            "iteration_cap": LGBM_CONFIG["n_estimators"],
            "hpo_round_cap_hit": bool(LGBM_BEST_ITERATION_HPO >= HPO_LGBM_MAX_ROUNDS),
            "final_iteration_cap": int(LGBM_CONFIG["n_estimators"]),
            "early_stopping_rounds": LGBM_EARLY_STOPPING,
            "training_rows": len(y_train),
            "validation_rows": len(y_val),
        },
        indent=2,
    )
)

# The previous full-data GPU run failed inside LightGBM's native split finder
# (`best_split_info.left_count > 0`). Use explicit, logged retries rather than
# allowing a multi-hour notebook run to terminate without a model artifact.
# Attempt 1 preserves the HPO-selected parameters and uses double-precision GPU
# histogram accumulation. Attempts 2/3 apply conservative tree constraints.
LGBM_TRAINING_ATTEMPTS = []
LGBM_BACKEND_USED = None
LGBM_ROBUSTNESS_OVERRIDES = {}
LGBM_TRAIN_SECONDS = 0.0

def _make_lgb_callbacks():
    # Fresh callback instances per retry prevent early-stopping state leaking
    # from a failed native fit into the next attempt.
    return [
        lgb.early_stopping(
            stopping_rounds=LGBM_EARLY_STOPPING,
            first_metric_only=True,
            verbose=True,
        ),
        lgb.log_evaluation(period=5 if DRY_RUN else 25),
    ]

_lgb_attempt_configs = [("primary_gpu_dp", dict(LGBM_CONFIG))]
if not DRY_RUN:
    _robust_gpu = dict(LGBM_CONFIG)
    _robust_gpu.update({
        "device_type": "gpu", "gpu_device_id": 0, "gpu_use_dp": True,
        "num_leaves": min(int(LGBM_CONFIG.get("num_leaves", 31)), 31),
        "min_child_samples": max(int(LGBM_CONFIG.get("min_child_samples", 20)), 500),
        "max_bin": min(int(LGBM_CONFIG.get("max_bin", 63)), 31),
        "feature_pre_filter": False,
        "force_col_wise": False,
    })
    _lgb_attempt_configs.append(("conservative_gpu_dp", _robust_gpu))
    _robust_cpu = dict(_robust_gpu)
    _robust_cpu.pop("gpu_device_id", None)
    _robust_cpu.pop("gpu_use_dp", None)
    _robust_cpu.update({"device_type": "cpu", "force_col_wise": True})
    _lgb_attempt_configs.append(("conservative_cpu_colwise", _robust_cpu))

_lgb_fit_start = time.time()
_lgb_last_error = None
for _attempt_name, _attempt_config in _lgb_attempt_configs:
    tee_print("\n[LightGBM attempt]", _attempt_name,
              "backend=" + str(_attempt_config.get("device_type", "cpu")),
              "max_bin=" + str(_attempt_config.get("max_bin")),
              "num_leaves=" + str(_attempt_config.get("num_leaves")),
              "min_child_samples=" + str(_attempt_config.get("min_child_samples")),
              flush=True)
    _attempt_start = time.time()
    _attempt_record = {
        "attempt": _attempt_name,
        "config": dict(_attempt_config),
        "started_at_unix": float(_attempt_start),
        "status": "running",
    }
    try:
        _candidate_model = LGBMClassifier(**_attempt_config)
        _candidate_model.fit(
            X_train,
            y_train,
            sample_weight=train_sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=_make_lgb_callbacks(),
        )
        lgb_model = _candidate_model
        LGBM_CONFIG = dict(_attempt_config)
        LGBM_BACKEND_USED = str(_attempt_config.get("device_type", "cpu"))
        LGBM_ROBUSTNESS_OVERRIDES = {
            key: {"selected_hpo_value": LGBM_BEST_PARAMS.get(key), "used_value": _attempt_config.get(key)}
            for key in ("num_leaves", "min_child_samples", "max_bin")
            if key in LGBM_BEST_PARAMS and _attempt_config.get(key) != LGBM_BEST_PARAMS.get(key)
        }
        _attempt_record.update({
            "status": "success",
            "elapsed_seconds": float(time.time() - _attempt_start),
            "best_iteration": int(getattr(lgb_model, "best_iteration_", 0) or _attempt_config["n_estimators"]),
        })
        LGBM_TRAINING_ATTEMPTS.append(_attempt_record)
        tee_print("[LightGBM attempt] SUCCESS:", _attempt_name,
                  "elapsed_seconds=", f"{_attempt_record['elapsed_seconds']:.1f}")
        break
    except Exception as _lgb_exc:
        _lgb_last_error = _lgb_exc
        _attempt_record.update({
            "status": "failed",
            "elapsed_seconds": float(time.time() - _attempt_start),
            "exception_type": type(_lgb_exc).__name__,
            "exception": str(_lgb_exc)[:4000],
        })
        LGBM_TRAINING_ATTEMPTS.append(_attempt_record)
        tee_print("[LightGBM attempt] FAILED:", _attempt_name, repr(_lgb_exc), flush=True)
        if "_candidate_model" in locals():
            del _candidate_model
        gc.collect()
        # Do not attempt a CPU fallback in diagnostic DRY_RUN mode.
        if DRY_RUN:
            break
else:
    # The for-loop's else is reached only if every configured attempt failed.
    pass

LGBM_TRAIN_SECONDS = float(time.time() - _lgb_fit_start)
with open(RUN_DIR / "lightgbm_training_attempts.json", "w", encoding="utf-8") as _f:
    json.dump({
        "attempts": LGBM_TRAINING_ATTEMPTS,
        "backend_used": LGBM_BACKEND_USED,
        "robustness_overrides": LGBM_ROBUSTNESS_OVERRIDES,
        "total_elapsed_seconds": LGBM_TRAIN_SECONDS,
        "success": LGBM_BACKEND_USED is not None,
    }, _f, indent=2)
if LGBM_BACKEND_USED is None:
    raise RuntimeError(
        "All configured LightGBM attempts failed. See "
        f"{RUN_DIR / 'lightgbm_training_attempts.json'} for each native exception. "
        f"Last error: {_lgb_last_error!r}"
    )

LGBM_BEST_ITERATION = int(
    getattr(
        lgb_model,
        "best_iteration_",
        LGBM_CONFIG["n_estimators"],
    )
)

if LGBM_BEST_ITERATION <= 0:
    LGBM_BEST_ITERATION = int(
        LGBM_CONFIG["n_estimators"]
    )

tee_print(
    "\nLightGBM training seconds:",
    f"{LGBM_TRAIN_SECONDS:.2f}"
)

tee_print(
    "Best iteration:",
    LGBM_BEST_ITERATION
)

# Validation evaluation.
t0 = time.time()

lgb_val_probs = fixed_class_probabilities(
    lgb_model,
    lgb_model.predict_proba(
        X_val,
        num_iteration=LGBM_BEST_ITERATION,
    ),
)

LGBM_VAL_SECONDS = (
    time.time() - t0
)

LGBM_VAL_METRICS, LGBM_VAL_CM = (
    save_metrics_bundle(
        "LightGBM",
        "validation",
        y_val,
        lgb_val_probs.argmax(axis=1),
        lgb_val_probs,
        {
            "best_iteration": LGBM_BEST_ITERATION,
            "training_seconds": LGBM_TRAIN_SECONDS,
            "prediction_seconds": LGBM_VAL_SECONDS,
            "train_rows": len(y_train),
        },
    )
)

# Save model + native model text.
joblib.dump(
    lgb_model,
    RUN_DIR / "lightgbm_baseline.joblib"
)

lgb_model.booster_.save_model(
    str(
        RUN_DIR
        / "lightgbm_baseline.txt"
    )
)

lgb_importance = (
    pd.DataFrame(
        {
            "feature":
                FEATURE_COLUMNS,

            "importance_gain":
                lgb_model.booster_
                .feature_importance(
                    importance_type="gain"
                ),

            "importance_split":
                lgb_model.booster_
                .feature_importance(
                    importance_type="split"
                ),
        }
    )
    .sort_values(
        "importance_gain",
        ascending=False,
    )
)

lgb_importance.to_csv(
    RUN_DIR
    / "lightgbm_feature_importance.csv",
    index=False,
)

with open(
    RUN_DIR / "lightgbm_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "config": LGBM_CONFIG,
            "hpo_budget_seconds_per_family": HPO_BUDGET_SECONDS_PER_FAMILY,
            "hpo_trials_completed": len(HPO_LGBM_RESULTS),
            "hpo_best_macro_f1": max((r["macro_f1"] for r in HPO_LGBM_RESULTS), default=None),
            "hpo_best_iteration_on_pilot": LGBM_BEST_ITERATION_HPO,
            "iteration_cap": LGBM_CONFIG["n_estimators"],
            "hpo_round_cap_hit": bool(LGBM_BEST_ITERATION_HPO >= HPO_LGBM_MAX_ROUNDS),
            "final_iteration_cap": int(LGBM_CONFIG["n_estimators"]),
            "early_stopping_rounds":
                LGBM_EARLY_STOPPING,
            "best_iteration":
                LGBM_BEST_ITERATION,
            "training_seconds":
                LGBM_TRAIN_SECONDS,
            "validation_seconds":
                LGBM_VAL_SECONDS,
            "train_rows":
                int(len(y_train)),
            "validation_rows":
                int(len(y_val)),
            "feature_count":
                N_FEATURES,
            "class_weighting":
                (
                    "inverse training frequency, "
                    "normalized to mean 1, "
                    "clipped [0.25,5.0]"
                ),
        },
        f,
        indent=2,
    )

del lgb_val_probs
del lgb_model
gc.collect()

tee_print("\nLightGBM baseline frozen.")
tee_print("TEST DATA HAS NOT BEEN READ.")

# Freeze the protocol before the test phase.
FROZEN_PROTOCOL = {
    "dry_run": DRY_RUN,
    "dataset": DATASET_NAME,
    "dataset_size": DATASET_SIZE,
    "n_classes": N_CLASSES,
    "n_features": N_FEATURES,
    "training_rows": int(len(y_train)),
    "validation_rows": int(len(y_val)),
    "train_sampling": (
        "none"
        if not DRY_RUN
        else "class-complete deterministic dry-run subset"
    ),
    "validation_sampling": (
        "uniform deterministic reservoir"
        if not DRY_RUN
        else "class-complete deterministic dry-run subset"
    ),
    "class_weighting": (
        "inverse training frequency, normalized to mean 1, clipped [0.25,5.0]"
    ),
    "lgbm_config": LGBM_CONFIG,
    "xgboost_config": XGB_CONFIG,
    "xgboost_external_memory": XGB_EXTERNAL_MEMORY,
    "may_accessed": False,
}

with open(
    RUN_DIR
    / "pass5_frozen_protocol.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        FROZEN_PROTOCOL,
        f,
        indent=2,
    )



In [ ]:
# ============================================================
# Cell 12 — COMMON APRIL-TRAINED STACKERS + DISJOINT AUDIT
# ============================================================

tee_print("=" * 80)
tee_print("SAME-POLICY APRIL STACKING / CALIBRATION")
tee_print("=" * 80)

lgb_model = joblib.load(RUN_DIR / "lightgbm_baseline.joblib")


def _raw_probabilities_for_model(model_name, X_source, batch_rows=8192):
    out = np.empty((len(X_source), N_CLASSES), dtype=np.float32)
    for _start in range(0, len(X_source), batch_rows):
        _stop = min(_start + batch_rows, len(X_source))
        _batch = np.asarray(X_source[_start:_stop], dtype=np.float32)
        if model_name == "LightGBM":
            _raw = lgb_model.predict_proba(_batch, num_iteration=LGBM_BEST_ITERATION)
            _raw = fixed_class_probabilities(lgb_model, _raw)
        elif model_name == "XGBoost OvR":
            _raw = xgb_ovr_predict_batch(
                XGB_OVR_MODELS, _batch, XGB_BEST_ITERATIONS, use_cuda=(not DRY_RUN)
            )
        else:
            raise ValueError(model_name)
        out[_start:_stop] = np.asarray(_raw, dtype=np.float32)
    return out


def _apply_multinomial_stacker(stacker, raw_probs):
    raw_probs = np.asarray(raw_probs, dtype=np.float32)
    if stacker is None:
        return raw_probs
    calibrated = stacker.predict_proba(raw_probs)
    return fixed_class_probabilities(stacker, calibrated).astype(np.float32, copy=False)


def predict_tree_probabilities(model_name, X_batch):
    raw = _raw_probabilities_for_model(model_name, X_batch, batch_rows=max(1, len(X_batch)))
    stacker = LGBM_STACKER if model_name == "LightGBM" else XGB_STACKER
    return _apply_multinomial_stacker(stacker, raw)


if DRY_RUN:
    LGBM_STACKER = None
    XGB_STACKER = None
    LGBM_CAL_AUDIT = XGB_CAL_AUDIT = None
    CALIBRATION_ACTIVE = False
    STACKER_CONFIG = {"enabled": False, "reason": "dry-run; not paper-valid"}
else:
    assert len(y_cal_fit) == 400_000 and len(y_cal_audit) == 100_000
    assert np.array_equal(np.unique(y_cal_fit), np.arange(N_CLASSES))
    assert np.array_equal(np.unique(y_cal_audit), np.arange(N_CLASSES))

    STACKER_CONFIG = {
        "method": "multinomial LogisticRegression on base-model 50-class probability vectors",
        "same_for_both_tree_families": True,
        "training_partition": "April 2025, disjoint stacker_fit partition",
        "training_rows": int(len(y_cal_fit)),
        "audit_partition": "April 2025, disjoint calibration_audit partition",
        "audit_rows": int(len(y_cal_audit)),
        "C": 1.0,
        "solver": "lbfgs",
        "max_iter": 250,
        "class_weight": None,
        "selection_using_audit": False,
    }

    _audit_rows = []
    _stack_fit_seconds = {}
    for _model_name in ("LightGBM", "XGBoost OvR"):
        tee_print(f"Building {_model_name} stacker-fit probabilities on 400,000 April flows...")
        _t0 = time.time()
        _fit_probs = _raw_probabilities_for_model(_model_name, X_cal_fit)
        _stacker = LogisticRegression(
            C=STACKER_CONFIG["C"], solver=STACKER_CONFIG["solver"],
            max_iter=STACKER_CONFIG["max_iter"], random_state=SEED,
        )
        _stacker.fit(_fit_probs, y_cal_fit)
        _stack_fit_seconds[_model_name] = float(time.time() - _t0)
        _stack_path = RUN_DIR / ("lightgbm_april_stacker.joblib" if _model_name == "LightGBM" else "xgboost_april_stacker.joblib")
        joblib.dump(_stacker, _stack_path)
        if _model_name == "LightGBM":
            LGBM_STACKER = _stacker
        else:
            XGB_STACKER = _stacker
        tee_print(f"{_model_name} stacker fit seconds={_stack_fit_seconds[_model_name]:.1f}; n_iter={int(np.max(_stacker.n_iter_))}; classes={len(_stacker.classes_)}")
        del _fit_probs, _stacker
        gc.collect()

    # Evaluate calibration on a disjoint April audit set; it is never used to tune the stacker.
    for _model_name in ("LightGBM", "XGBoost OvR"):
        _raw_audit = _raw_probabilities_for_model(_model_name, X_cal_audit)
        _stacker = LGBM_STACKER if _model_name == "LightGBM" else XGB_STACKER
        _cal_audit = _apply_multinomial_stacker(_stacker, _raw_audit)
        _raw_metrics = metrics_from_predictions(y_cal_audit, _raw_audit.argmax(axis=1), _raw_audit)
        _cal_metrics = metrics_from_predictions(y_cal_audit, _cal_audit.argmax(axis=1), _cal_audit)
        _audit_rows.append({
            "model": _model_name,
            "audit_rows": int(len(y_cal_audit)),
            "raw_macro_f1": _raw_metrics["macro_f1"],
            "stacked_macro_f1": _cal_metrics["macro_f1"],
            "raw_ece_10": _raw_metrics["ece_10"],
            "stacked_ece_10": _cal_metrics["ece_10"],
            "raw_accuracy": _raw_metrics["accuracy"],
            "stacked_accuracy": _cal_metrics["accuracy"],
            "stacker_fit_seconds": _stack_fit_seconds[_model_name],
            "audit_used_for_selection": False,
        })
        del _raw_audit, _cal_audit, _raw_metrics, _cal_metrics
        gc.collect()

    CAL_AUDIT_DF = pd.DataFrame(_audit_rows)
    CAL_AUDIT_DF.to_csv(RUN_DIR / "pass5_tree_calibration_audit.csv", index=False)
    with open(RUN_DIR / "pass5_stacker_config.json", "w", encoding="utf-8") as f:
        json.dump({"config": STACKER_CONFIG, "fit_seconds": _stack_fit_seconds, "audit_results": _audit_rows}, f, indent=2)
    tee_print("Disjoint April calibration audit:")
    tee_print(CAL_AUDIT_DF.to_string(index=False))

# Rewrite the frozen protocol only after both base models and both post-processors are frozen.
FROZEN_PROTOCOL = {
    "dry_run": DRY_RUN,
    "dataset": DATASET_NAME,
    "dataset_size": DATASET_SIZE,
    "n_classes": N_CLASSES,
    "n_features": N_FEATURES,
    "training_rows": int(len(y_train)),
    "train_dates": [TRAIN_DATES[0], TRAIN_DATES[-1]],
    "early_stopping_rows": int(len(y_val)),
    "lgbm_config": LGBM_CONFIG,
    "xgboost_config": XGB_CONFIG,
    "hpo_summary": HPO_FAMILY_SUMMARY,
    "calibration_policy": STACKER_CONFIG,
    "may_accessed": False,
    "frozen_before_test_access": True,
    "neural_reference_status": "external point-estimate only; provenance and metric discrepancy remain unresolved",
}
with open(RUN_DIR / "pass5_frozen_protocol.json", "w", encoding="utf-8") as f:
    json.dump(FROZEN_PROTOCOL, f, indent=2)

# Free April partitions once stackers and audit outputs have been persisted.
if not DRY_RUN:
    for _tmp_name in ("X_cal_fit", "y_cal_fit", "X_cal_audit", "y_cal_audit", "X_hpo_val", "y_hpo_val"):
        globals().pop(_tmp_name, None)
    gc.collect()

tee_print("Both stackers and the validation protocol are frozen; May has NOT been read.")

In [ ]:
# ============================================================
# Cell 13 — MAY TEST FIREWALL, CALIBRATED METRICS + FAIR TREE INFERENCE
# ============================================================

tee_print("=" * 80)
tee_print("MAY 2025 TEST EVALUATION — FIRST ALLOWED TEST ACCESS")
tee_print("=" * 80)

TEST_BATCH_SIZE = 512
assert (RUN_DIR / "lightgbm_baseline.joblib").exists()
assert (RUN_DIR / "xgboost_baseline.json").exists()
assert (RUN_DIR / "pass5_frozen_protocol.json").exists()
with open(RUN_DIR / "pass5_frozen_protocol.json", "r", encoding="utf-8") as f:
    frozen = json.load(f)
assert frozen["may_accessed"] is False and frozen["frozen_before_test_access"] is True

preprocessor = joblib.load(RUN_DIR / "preprocessor.joblib")

if DRY_RUN:
    X_test, y_test = class_quota_sample_period(TEST_DATES, DRY_TEST_PER_CLASS, TEST_BENCHMARK_SEED, "DRY-TEST")
    test_day_ids = np.zeros(len(y_test), dtype=np.uint8)
    TEST_DAY_NAMES = ["DRY_SAMPLE_NOT_DAYS"]
    TEST_ACTUAL_ROWS = len(y_test)
else:
    TEST_X_PATH = SCRATCH_DIR / "X_test.npy"
    TEST_Y_PATH = SCRATCH_DIR / "y_test.npy"
    TEST_DAY_ID_PATH = SCRATCH_DIR / "test_day_ids.npy"
    for _p in (TEST_X_PATH, TEST_Y_PATH, TEST_DAY_ID_PATH):
        if _p.exists():
            _p.unlink()

    X_test_mm = np.lib.format.open_memmap(TEST_X_PATH, mode="w+", dtype=np.float32, shape=(TEST_EXPECTED_ROWS, N_FEATURES))
    y_test_mm = np.lib.format.open_memmap(TEST_Y_PATH, mode="w+", dtype=np.int16, shape=(TEST_EXPECTED_ROWS,))
    day_mm = np.lib.format.open_memmap(TEST_DAY_ID_PATH, mode="w+", dtype=np.uint8, shape=(TEST_EXPECTED_ROWS,))
    write_pos = 0
    t0 = time.time()
    TEST_DAY_NAMES = list(TEST_DATES)
    tee_print("Building disk-backed May test data and recording true day-block IDs...")

    for day_id, date_str in enumerate(TEST_DATES):
        for raw in iter_period_chunks([date_str]):
            X_chunk, y_chunk = raw_to_features(raw)
            n = len(y_chunk)
            stop = write_pos + n
            if stop > TEST_EXPECTED_ROWS:
                raise RuntimeError("May known-class row count exceeded the fixed expected count.")
            if n:
                X_test_mm[write_pos:stop] = X_chunk
                y_test_mm[write_pos:stop] = y_chunk
                day_mm[write_pos:stop] = np.uint8(day_id)
            write_pos = stop
            del X_chunk, y_chunk, raw
        if day_id % 5 == 0 or day_id == len(TEST_DATES) - 1:
            tee_print(f"[TEST BUILD] day {day_id+1}/{len(TEST_DATES)}; rows={write_pos:,}; elapsed={(time.time()-t0)/60:.2f} min")

    assert write_pos == TEST_EXPECTED_ROWS, f"Expected {TEST_EXPECTED_ROWS:,} test rows, found {write_pos:,}."
    X_test_mm.flush(); y_test_mm.flush(); day_mm.flush()
    del X_test_mm, y_test_mm, day_mm
    gc.collect()
    X_test = np.load(TEST_X_PATH, mmap_mode="r+")
    y_test = np.load(TEST_Y_PATH, mmap_mode="r")
    test_day_ids = np.load(TEST_DAY_ID_PATH, mmap_mode="r")
    TEST_ACTUAL_ROWS = int(len(y_test))
    assert len(test_day_ids) == TEST_ACTUAL_ROWS

tee_print("Applying the frozen training-fitted preprocessing to May...")
preprocessor.transform_inplace(X_test)
if hasattr(X_test, "flush"):
    X_test.flush()
assert_finite_chunked(X_test)
assert np.array_equal(np.unique(y_test), np.arange(N_CLASSES))
tee_print("May known-class rows:", f"{TEST_ACTUAL_ROWS:,}")
tee_print("May day blocks:", len(TEST_DAY_NAMES))


def _ece_accumulate(counts, conf_sums, correct_sums, y_true, probs):
    pred = probs.argmax(axis=1)
    confidence = probs.max(axis=1)
    correct = pred == y_true
    bins = np.minimum((confidence * 10).astype(np.int16), 9)
    for b in range(10):
        mask = bins == b
        if mask.any():
            counts[b] += int(mask.sum())
            conf_sums[b] += float(confidence[mask].sum())
            correct_sums[b] += int(correct[mask].sum())


def _ece_finalize(counts, conf_sums, correct_sums):
    total = int(counts.sum())
    if total == 0:
        return float("nan")
    ece = 0.0
    for b in range(10):
        if counts[b] == 0:
            continue
        ece += (counts[b] / total) * abs(correct_sums[b] / counts[b] - conf_sums[b] / counts[b])
    return float(ece)


def _evaluate_model_on_test(model_name):
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    daily_cm = np.zeros((len(TEST_DAY_NAMES), N_CLASSES, N_CLASSES), dtype=np.int64)
    preds_store = np.empty(TEST_ACTUAL_ROWS, dtype=np.int16)
    ece_counts = np.zeros(10, dtype=np.int64)
    ece_conf_sum = np.zeros(10, dtype=np.float64)
    ece_correct_sum = np.zeros(10, dtype=np.int64)

    t0_eval = time.time()
    for start in range(0, TEST_ACTUAL_ROWS, 8192):
        stop = min(start + 8192, TEST_ACTUAL_ROWS)
        X_batch = np.asarray(X_test[start:stop], dtype=np.float32)
        y_batch = np.asarray(y_test[start:stop], dtype=np.int16)
        days_batch = np.asarray(test_day_ids[start:stop], dtype=np.int16)
        probs = predict_tree_probabilities(model_name, X_batch)
        pred = probs.argmax(axis=1).astype(np.int16)
        np.add.at(cm, (y_batch, pred), 1)
        preds_store[start:stop] = pred
        _ece_accumulate(ece_counts, ece_conf_sum, ece_correct_sum, y_batch, probs)
        if not DRY_RUN:
            for day in np.unique(days_batch):
                mask = days_batch == day
                np.add.at(daily_cm[int(day)], (y_batch[mask], pred[mask]), 1)
        elif len(TEST_DAY_NAMES) == 1:
            np.add.at(daily_cm[0], (y_batch, pred), 1)
        if stop % 250_000 < 8192 or stop == TEST_ACTUAL_ROWS:
            tee_print(f"[{model_name}] evaluated {stop:,}/{TEST_ACTUAL_ROWS:,} rows")

    metrics = metrics_from_confusion_matrix(cm)
    metrics["ece_10"] = _ece_finalize(ece_counts, ece_conf_sum, ece_correct_sum)
    metrics.update({
        "model": model_name,
        "split": "May 2025 full known-class population" if not DRY_RUN else "dry-run subset (not paper-valid)",
        "samples": int(TEST_ACTUAL_ROWS),
        "calibrated_with_april_stacker": bool(CALIBRATION_ACTIVE),
        "evaluation_seconds": float(time.time() - t0_eval),
    })

    # Optimized-model-stack inference only, including the same LR stacker recipe
    # for both tree families; preprocessing and HDF5 I/O are excluded.
    bench_batches = DRY_BENCHMARK_BATCHES if DRY_RUN else 50
    required = TEST_BATCH_SIZE * (5 + bench_batches)
    if TEST_ACTUAL_ROWS < required:
        raise RuntimeError(f"Insufficient test rows for benchmark: need {required}, have {TEST_ACTUAL_ROWS}.")
    for i in range(5):
        _ = predict_tree_probabilities(model_name, np.asarray(X_test[i*TEST_BATCH_SIZE:(i+1)*TEST_BATCH_SIZE], dtype=np.float32))
    timings = []
    for i in range(bench_batches):
        start = (i + 5) * TEST_BATCH_SIZE
        batch = np.asarray(X_test[start:start+TEST_BATCH_SIZE], dtype=np.float32)
        t0_b = time.perf_counter()
        _ = predict_tree_probabilities(model_name, batch)
        timings.append(time.perf_counter() - t0_b)
    benchmark = {
        "model": model_name, "batch_size": TEST_BATCH_SIZE,
        "warmup_batches": 5, "measured_batches": bench_batches,
        "latency_ms_per_batch": float(np.mean(timings) * 1000),
        "throughput_flows_per_sec": float(bench_batches * TEST_BATCH_SIZE / np.sum(timings)),
        "scope": "tree inference plus April-trained multinomial logistic stacker; feature preprocessing and HDF5 I/O excluded",
        "comparable_only_with_other_Pass5_tree_rows": True,
    }
    slug = "lightgbm" if model_name == "LightGBM" else "xgboost_ovr"
    np.save(RUN_DIR / f"{slug}_test_predictions.npy", preds_store)
    np.save(RUN_DIR / f"{slug}_test_confusion_matrix.npy", cm)
    np.save(RUN_DIR / f"{slug}_test_daily_confusion_matrices.npy", daily_cm)
    with open(RUN_DIR / f"{slug}_test_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    with open(RUN_DIR / f"{slug}_inference_benchmark.json", "w", encoding="utf-8") as f:
        json.dump(benchmark, f, indent=2)
    tee_print(f"{model_name} May metrics:", json.dumps(metrics, indent=2))
    tee_print(f"{model_name} benchmark:", json.dumps(benchmark, indent=2))
    return cm, daily_cm, metrics, benchmark, preds_store


LGBM_TEST_CM, LGBM_DAILY_CM, LGBM_TEST_METRICS, LGBM_BENCHMARK, LGBM_TEST_PREDICTIONS = _evaluate_model_on_test("LightGBM")
XGB_TEST_CM, XGB_DAILY_CM, XGB_TEST_METRICS, XGB_BENCHMARK, XGB_TEST_PREDICTIONS = _evaluate_model_on_test("XGBoost OvR")

np.save(RUN_DIR / "may_test_labels.npy", np.asarray(y_test, dtype=np.int16))
np.save(RUN_DIR / "may_test_day_ids.npy", np.asarray(test_day_ids, dtype=np.uint8))
np.save(RUN_DIR / "may_test_confusion_matrix_support.npy", LGBM_TEST_CM.sum(axis=1).astype(np.int64))

# Persist point estimates from the imported neural reference, but never attach
# a bootstrap CI to it without neural per-flow/day predictions.
NEURAL_REFERENCE_WARNING = {
    "model": f"Neural {NEURAL_SELECTED_METHOD}",
    "status": "external point-estimate only",
    "reason": "Pass 5 has only aggregate neural test metrics/confusion matrix; metric/provenance mismatch with supplied Pass-3 notebook must be resolved before cross-family inferential claims.",
    "neural_bootstrap_ci_computed": False,
    "neural_inference_speed_compared": False,
}
with open(RUN_DIR / "neural_reference_provenance_warning.json", "w", encoding="utf-8") as f:
    json.dump(NEURAL_REFERENCE_WARNING, f, indent=2)

# Only now update the frozen protocol to record the test population as accessed.
FROZEN_PROTOCOL["may_accessed"] = True
FROZEN_PROTOCOL["test_rows"] = int(TEST_ACTUAL_ROWS)
FROZEN_PROTOCOL["test_day_blocks"] = int(len(TEST_DAY_NAMES)) if not DRY_RUN else None
FROZEN_PROTOCOL["test_metrics_frozen_after_protocol"] = True
with open(RUN_DIR / "pass5_frozen_protocol.json", "w", encoding="utf-8") as f:
    json.dump(FROZEN_PROTOCOL, f, indent=2)

tee_print("May test evaluation complete. Neural confidence intervals remain intentionally omitted.")

In [ ]:
# ============================================================
# Cell 14 — PER-CLASS RESULTS (NEURAL ROW IS DESCRIPTIVE ONLY)
# ============================================================

def per_class_from_cm(cm, prefix):
    cm = np.asarray(cm, dtype=np.int64)
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1).astype(np.int64)
    predicted = cm.sum(axis=0).astype(np.float64)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros_like(tp), where=(precision+recall)>0)
    return pd.DataFrame({"class": KNOWN_APPS, f"{prefix}_support": support,
                         f"{prefix}_precision": precision, f"{prefix}_recall": recall, f"{prefix}_f1": f1})

neural_pc = per_class_from_cm(NEURAL_CM, "neural_external")
lgb_pc = per_class_from_cm(LGBM_TEST_CM, "lgbm")
xgb_pc = per_class_from_cm(XGB_TEST_CM, "xgb_ovr")
per_class_df = neural_pc.merge(lgb_pc, on="class", validate="one_to_one").merge(xgb_pc, on="class", validate="one_to_one")
per_class_df["neural_comparison_status"] = "descriptive only; external reference provenance not yet reconciled"
per_class_df.to_csv(RUN_DIR / "pass5_per_class_comparison.csv", index=False)
tee_print("Per-class table saved:")
tee_print(per_class_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# Cell 15 — PAIRED DAY-BLOCK BOOTSTRAP FOR TREE BASELINES ONLY
# ============================================================

if DRY_RUN:
    tee_print("Skipping day-block bootstrap in DRY_RUN; no paper-valid intervals are produced.")
    bootstrap_ci_df = pd.DataFrame(columns=["model", "metric", "point_estimate", "ci95_low", "ci95_high", "bootstrap_unit", "n_blocks", "replicates"])
    tree_difference_df = pd.DataFrame(columns=["comparison", "metric", "point_difference", "ci95_low", "ci95_high", "method", "bootstrap_unit", "n_blocks", "replicates"])
    bootstrap_ci_df.to_csv(RUN_DIR / "pass5_bootstrap_95ci.csv", index=False)
    tree_difference_df.to_csv(RUN_DIR / "pass5_tree_difference_block_bootstrap.csv", index=False)
else:
    assert LGBM_DAILY_CM.shape[0] == len(TEST_DATES)
    assert XGB_DAILY_CM.shape[0] == len(TEST_DATES)
    _n_days = len(TEST_DATES)
    _rng = np.random.default_rng(BOOTSTRAP_SEED)
    _metrics = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]
    _model_daily = {"LightGBM": LGBM_DAILY_CM, "XGBoost OvR": XGB_DAILY_CM}
    _model_point_cm = {"LightGBM": LGBM_TEST_CM, "XGBoost OvR": XGB_TEST_CM}
    _boot_values = {m: {metric: np.empty(BOOTSTRAP_REPS, dtype=np.float64) for metric in _metrics} for m in _model_daily}
    _diff_values = {metric: np.empty(BOOTSTRAP_REPS, dtype=np.float64) for metric in _metrics}

    for _rep in range(BOOTSTRAP_REPS):
        _sampled_days = _rng.integers(0, _n_days, size=_n_days)
        _day_weights = np.bincount(_sampled_days, minlength=_n_days).astype(np.int64)
        _cm_by_model = {}
        for _name, _daily in _model_daily.items():
            _cm = np.tensordot(_day_weights, _daily, axes=(0, 0)).astype(np.int64, copy=False)
            _cm_by_model[_name] = _cm
            _m = metrics_from_confusion_matrix(_cm)
            for _metric in _metrics:
                _boot_values[_name][_metric][_rep] = _m[_metric]
        _m_l = metrics_from_confusion_matrix(_cm_by_model["LightGBM"])
        _m_x = metrics_from_confusion_matrix(_cm_by_model["XGBoost OvR"])
        for _metric in _metrics:
            _diff_values[_metric][_rep] = _m_l[_metric] - _m_x[_metric]

    _rows = []
    for _name in _model_daily:
        _point = metrics_from_confusion_matrix(_model_point_cm[_name])
        for _metric in _metrics:
            _values = _boot_values[_name][_metric]
            _rows.append({
                "model": _name, "metric": _metric, "point_estimate": float(_point[_metric]),
                "ci95_low": float(np.quantile(_values, 0.025)), "ci95_high": float(np.quantile(_values, 0.975)),
                "bootstrap_unit": "May calendar day (paired block bootstrap)", "n_blocks": _n_days,
                "replicates": BOOTSTRAP_REPS,
            })
    bootstrap_ci_df = pd.DataFrame(_rows)
    bootstrap_ci_df.to_csv(RUN_DIR / "pass5_bootstrap_95ci.csv", index=False)

    _diff_rows = []
    _point_l = metrics_from_confusion_matrix(LGBM_TEST_CM)
    _point_x = metrics_from_confusion_matrix(XGB_TEST_CM)
    for _metric in _metrics:
        _values = _diff_values[_metric]
        _diff_rows.append({
            "comparison": "LightGBM - XGBoost OvR", "metric": _metric,
            "point_difference": float(_point_l[_metric] - _point_x[_metric]),
            "ci95_low": float(np.quantile(_values, 0.025)), "ci95_high": float(np.quantile(_values, 0.975)),
            "method": "paired daily-block bootstrap; same sampled days for both models",
            "bootstrap_unit": "May calendar day", "n_blocks": _n_days, "replicates": BOOTSTRAP_REPS,
        })
    tree_difference_df = pd.DataFrame(_diff_rows)
    tree_difference_df.to_csv(RUN_DIR / "pass5_tree_difference_block_bootstrap.csv", index=False)

    # Explicit status file prevents a point-estimate gap being mistaken for a CI.
    neural_diff_status = pd.DataFrame([
        {"comparison": "LightGBM - Neural reference", "status": "NOT_COMPUTED",
         "reason": "No neural per-flow/day-block predictions in the available Pass 5 artifacts; aggregate confusion matrix cannot support paired day-block bootstrap."},
        {"comparison": "XGBoost OvR - Neural reference", "status": "NOT_COMPUTED",
         "reason": "No neural per-flow/day-block predictions in the available Pass 5 artifacts; aggregate confusion matrix cannot support paired day-block bootstrap."},
    ])
    neural_diff_status.to_csv(RUN_DIR / "pass5_neural_difference_ci_status.csv", index=False)
    tee_print("Day-block model CIs:")
    tee_print(bootstrap_ci_df.to_string(index=False))
    tee_print("Paired tree-family differences:")
    tee_print(tree_difference_df.to_string(index=False))
    tee_print("Neural CI status:", neural_diff_status.to_dict(orient="records"))

In [ ]:
# ============================================================
# Cell 16 — RESULTS, CLAIM BOUNDARIES + FINAL MANIFEST
# ============================================================

comparison_rows = [
    {
        "model": f"Neural {NEURAL_SELECTED_METHOD} (external reference)",
        "role": "external point-estimate only; not retrained in Pass 5",
        "comparable_tree_protocol": False,
        "training_flows": TRAIN_EXPECTED_ROWS,
        "test_flows": NEURAL_TEST_FLOWS,
        "accuracy": NEURAL_ACCURACY,
        "macro_precision": NEURAL_MACRO_PRECISION,
        "macro_recall": NEURAL_MACRO_RECALL,
        "macro_f1": NEURAL_MACRO_F1,
        "weighted_f1": NEURAL_WEIGHTED_F1,
        "ece_10": NEURAL_ECE,
        "latency_ms_per_batch": None,
        "throughput_flows_per_sec": None,
        "calibration": "external neural stacker; provenance/metric discrepancy unresolved",
        "hpo_seconds": None,
        "final_fit_seconds": None,
        "inferential_ci": "not computed",
    },
    {
        "model": "LightGBM multiclass + April LR stacker",
        "role": "Pass 5 tuned tree baseline",
        "comparable_tree_protocol": True,
        "training_flows": int(len(y_train)), "test_flows": int(LGBM_TEST_METRICS["samples"]),
        "accuracy": LGBM_TEST_METRICS["accuracy"], "macro_precision": LGBM_TEST_METRICS["macro_precision"],
        "macro_recall": LGBM_TEST_METRICS["macro_recall"], "macro_f1": LGBM_TEST_METRICS["macro_f1"],
        "weighted_f1": LGBM_TEST_METRICS["weighted_f1"], "ece_10": LGBM_TEST_METRICS["ece_10"],
        "latency_ms_per_batch": LGBM_BENCHMARK["latency_ms_per_batch"],
        "throughput_flows_per_sec": LGBM_BENCHMARK["throughput_flows_per_sec"],
        "calibration": "same April multinomial LR stacker recipe as XGBoost OvR",
        "hpo_seconds": float(HPO_FAMILY_SUMMARY.get("LightGBM", {}).get("elapsed_seconds", 0.0)),
        "final_fit_seconds": float(LGBM_TRAIN_SECONDS),
        "inferential_ci": "paired day-block bootstrap",
    },
    {
        "model": "XGBoost OvR + April LR stacker",
        "role": "Pass 5 tuned tree baseline",
        "comparable_tree_protocol": True,
        "training_flows": int(len(y_train)), "test_flows": int(XGB_TEST_METRICS["samples"]),
        "accuracy": XGB_TEST_METRICS["accuracy"], "macro_precision": XGB_TEST_METRICS["macro_precision"],
        "macro_recall": XGB_TEST_METRICS["macro_recall"], "macro_f1": XGB_TEST_METRICS["macro_f1"],
        "weighted_f1": XGB_TEST_METRICS["weighted_f1"], "ece_10": XGB_TEST_METRICS["ece_10"],
        "latency_ms_per_batch": XGB_BENCHMARK["latency_ms_per_batch"],
        "throughput_flows_per_sec": XGB_BENCHMARK["throughput_flows_per_sec"],
        "calibration": "same April multinomial LR stacker recipe as LightGBM",
        "hpo_seconds": float(HPO_FAMILY_SUMMARY.get("XGBoost_OvR", {}).get("elapsed_seconds", 0.0)),
        "final_fit_seconds": float(XGB_TRAIN_SECONDS),
        "inferential_ci": "paired day-block bootstrap",
    },
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(RUN_DIR / "pass5_final_comparison.csv", index=False)
_comparison_records = comparison_df.to_dict(orient="records")
for _record in _comparison_records:
    for _key, _value in list(_record.items()):
        if pd.isna(_value):
            _record[_key] = None
        elif isinstance(_value, (np.integer,)):
            _record[_key] = int(_value)
        elif isinstance(_value, (np.floating,)):
            _record[_key] = float(_value)
with open(RUN_DIR / "pass5_final_comparison.json", "w", encoding="utf-8") as f:
    json.dump(_comparison_records, f, indent=2, allow_nan=False)

# Final configs also state round caps and cap-hit status plainly.
with open(RUN_DIR / "lightgbm_config.json", "r", encoding="utf-8") as f:
    _lgb_cfg = json.load(f)
_lgb_cfg.update({
    "hpo_summary": HPO_FAMILY_SUMMARY.get("LightGBM", {}),
    "round_cap_hit": bool(LGBM_BEST_ITERATION >= int(LGBM_CONFIG["n_estimators"])),
    "cap_warning": bool(LGBM_BEST_ITERATION >= int(LGBM_CONFIG["n_estimators"])),
})
with open(RUN_DIR / "lightgbm_config.json", "w", encoding="utf-8") as f:
    json.dump(_lgb_cfg, f, indent=2)
with open(RUN_DIR / "xgboost_config.json", "r", encoding="utf-8") as f:
    _xgb_cfg = json.load(f)
_xgb_cfg.update({
    "hpo_summary": HPO_FAMILY_SUMMARY.get("XGBoost_OvR", {}),
    "round_cap_hit_fraction": float(np.mean(np.asarray(XGB_BEST_ITERATIONS) >= (XGB_N_ESTIMATORS - 1))),
    "cap_warning": bool(np.mean(np.asarray(XGB_BEST_ITERATIONS) >= (XGB_N_ESTIMATORS - 1)) > 0.2),
})
with open(RUN_DIR / "xgboost_config.json", "w", encoding="utf-8") as f:
    json.dump(_xgb_cfg, f, indent=2)

pass5_elapsed_seconds = float(time.time() - PASS5_RUN_START_EPOCH)

if DRY_RUN:
    bootstrap_summary = "Dry-run: day-block bootstrap skipped; all reported scores are diagnostic only."
else:
    _bf = bootstrap_ci_df[(bootstrap_ci_df.model == "LightGBM") & (bootstrap_ci_df.metric == "macro_f1")].iloc[0]
    _bx = bootstrap_ci_df[(bootstrap_ci_df.model == "XGBoost OvR") & (bootstrap_ci_df.metric == "macro_f1")].iloc[0]
    _bd = tree_difference_df[tree_difference_df.metric == "macro_f1"].iloc[0]
    bootstrap_summary = (
        f"LightGBM macro-F1: {_bf.point_estimate:.4f} [{_bf.ci95_low:.4f}, {_bf.ci95_high:.4f}] (day-block CI).\n"
        f"XGBoost OvR macro-F1: {_bx.point_estimate:.4f} [{_bx.ci95_low:.4f}, {_bx.ci95_high:.4f}] (day-block CI).\n"
        f"Paired LightGBM − XGBoost OvR macro-F1 difference: {_bd.point_difference:.4f} [{_bd.ci95_low:.4f}, {_bd.ci95_high:.4f}]."
    )

result_text = f"""# Pass 5 v10 — validation-controlled tree baselines

## Dataset protocol
- Dataset: {DATASET_NAME}-{DATASET_SIZE}; 50 known classes; 101 metadata-derived features.
- Train: {len(y_train):,} known-class flows, June 2024–March 2025.
- April validation reservoir: {VAL_SAMPLES_FULL:,} flows split into disjoint 30,000 HPO / 70,000 early-stop / 400,000 stacker-fit / 100,000 calibration-audit partitions.
- Final test: {TEST_ACTUAL_ROWS:,} known-class flows from May 2025 (full population in full mode).
- No payload bytes or SNI are included in the model feature vector. The 11 auxiliary flow statistics include full-flow values, so this is not a strict first-30-packets online-only classifier.

## Hyperparameter budget and selected models
- Both model families were tuned against the same 30k April HPO partition using the same 100k class-balanced training pilot, the same training-derived per-flow class weights with pilot sampling correction, and macro-F1 as the configuration-selection metric.
- Target HPO budget was {HPO_BUDGET_SECONDS_PER_FAMILY} seconds per family; actual completed trials and elapsed time are recorded in `pass5_hpo_summary.json` and `pass5_hpo_trials.csv`.
- LightGBM final round cap: {LGBM_CONFIG['n_estimators']}; best final iteration: {LGBM_BEST_ITERATION}; cap hit: {bool(LGBM_BEST_ITERATION >= LGBM_CONFIG['n_estimators'])}.
- XGBoost OvR final cap: {XGB_N_ESTIMATORS} rounds per class; median best iteration: {int(np.median(XGB_BEST_ITERATIONS))}; cap-hit fraction: {float(np.mean(np.asarray(XGB_BEST_ITERATIONS) >= (XGB_N_ESTIMATORS-1))):.3f}.
- Both final fits used the complete training population and the same 70k April early-stopping partition. Both use the same 400k April-trained multinomial logistic-regression stacker policy, audited on the disjoint 100k April partition.

## May test point estimates
```\n{comparison_df[['model','accuracy','macro_f1','weighted_f1','ece_10','latency_ms_per_batch','throughput_flows_per_sec']].to_string(index=False)}\n```

## Day-block uncertainty for tree models
{bootstrap_summary}

Bootstrap resampling unit: calendar day (31 test blocks), paired by using the same sampled days for LightGBM and XGBoost OvR; replicates={BOOTSTRAP_REPS}. This estimates day-to-day test-population variability, not seed/training variance. Only one fit per family was run.

## External neural reference — claim boundary
The historical `{NEURAL_SELECTED_METHOD}` result is retained as an external point-estimate reference only. Its available Pass 5 metrics/confusion matrix do not resolve the known discrepancy with the supplied Pass-3 notebook; per-flow/day-level neural predictions are unavailable. Therefore no neural bootstrap CI, neural-vs-tree significance claim, or cross-family throughput conclusion is produced. A same-pipeline neural rerun and provenance reconciliation remain mandatory before asserting neural architectural superiority.

## Remaining limitations
1. The LightGBM/XGBoost final round caps are bounded for Kaggle's 12-hour session limit. Any cap hit means convergence beyond that cap remains unresolved.
2. This is one random seed per model family. Day-block CIs do not replace multi-seed training variance.
3. The benchmark is closed-set over the 50 known application classes; the two background labels are excluded from the task.
4. The auxiliary features include full-flow statistics, so the model should not be described as strictly online/early-packet-only.

Total Pass 5 wall-clock elapsed: {pass5_elapsed_seconds/3600:.2f} hours.
"""
(RUN_DIR / "Pass5_paper_results.md").write_text(result_text, encoding="utf-8")

paper_paragraph = (
    f"We re-evaluated LightGBM multiclass and XGBoost one-vs-rest baselines on the same {len(y_train):,}-flow June 2024–March 2025 training population and the complete {TEST_ACTUAL_ROWS:,}-flow May 2025 known-class test population, using the same 101-feature representation, training-fitted preprocessing and train-derived per-flow weights. Both families received a fixed 15-minute HPO target on the same 100,000-flow class-balanced pilot and 30,000-flow April HPO partition; final early stopping used a disjoint 70,000-flow April partition. Each tree family used the same multinomial logistic-regression stacking policy trained on a separate 400,000-flow April partition and audited on 100,000 disjoint April flows. Uncertainty for the tree-family comparison is estimated with paired calendar-day block bootstrap intervals. The existing neural result is reported as an external point-estimate reference only because its provenance/metrics are not yet reconciled with the supplied Pass-3 notebook, and no neural per-flow/day predictions are available for paired inference. Consequently, these experiments support a controlled comparison between the two Pass 5 tree baselines, but do not yet justify a definitive cross-family claim of neural architectural superiority."
)
(RUN_DIR / "Pass5_paper_paragraph.txt").write_text(paper_paragraph, encoding="utf-8")

manifest = {
    "notebook_version": "Pass 5 v10",
    "dry_run": DRY_RUN,
    "dataset": DATASET_NAME,
    "dataset_size": DATASET_SIZE,
    "n_classes": N_CLASSES,
    "feature_dimension": N_FEATURES,
    "train_dates": [TRAIN_DATES[0], TRAIN_DATES[-1]],
    "train_rows": int(len(y_train)),
    "april_partition": PARTITION_MANIFEST if not DRY_RUN else {"dry_run": True, "disjoint": False},
    "test": {"dates": [TEST_DATES[0], TEST_DATES[-1]], "actual_flows": int(TEST_ACTUAL_ROWS), "day_blocks": len(TEST_DAY_NAMES) if not DRY_RUN else None},
    "hpo": HPO_FAMILY_SUMMARY,
    "final_models": {
        "LightGBM": {"config": LGBM_CONFIG, "best_iteration": int(LGBM_BEST_ITERATION), "round_cap_hit": bool(LGBM_BEST_ITERATION >= LGBM_CONFIG["n_estimators"]), "fit_seconds": float(LGBM_TRAIN_SECONDS)},
        "XGBoost_OvR": {"config": XGB_CONFIG, "best_iteration_median": int(np.median(XGB_BEST_ITERATIONS)), "round_cap_hit_fraction": float(np.mean(np.asarray(XGB_BEST_ITERATIONS) >= (XGB_N_ESTIMATORS-1))), "fit_seconds": float(XGB_TRAIN_SECONDS)},
    },
    "calibration": STACKER_CONFIG,
    "test_metrics": {"LightGBM": LGBM_TEST_METRICS, "XGBoost_OvR": XGB_TEST_METRICS},
    "bootstrap": {"unit": "calendar-day block, paired between tree models", "blocks": len(TEST_DAY_NAMES) if not DRY_RUN else None, "replicates": BOOTSTRAP_REPS, "neural_ci_computed": False},
    "neural_reference": NEURAL_REFERENCE_WARNING,
    "inference_scope": "tree predictions + April-trained LR stacker only; preprocessing/HDF5 excluded; not compared to neural throughput",
    "reproducibility_status": "tree-family comparison materially controlled; neural cross-family claim still requires same-pipeline retraining and metric/provenance reconciliation",
    "total_wallclock_seconds": pass5_elapsed_seconds,
    "available_libraries": {name: importlib_metadata.version(name) for name in ("numpy", "pandas", "scikit-learn", "lightgbm", "xgboost", "joblib", "tables")},
}
with open(RUN_DIR / "pass5_final_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

tee_print("Final comparison (neural row is external and not inferentially comparable):")
tee_print(comparison_df.to_string(index=False))
tee_print("Results report:", RUN_DIR / "Pass5_paper_results.md")
tee_print("Paper paragraph:", RUN_DIR / "Pass5_paper_paragraph.txt")
tee_print("Manifest:", RUN_DIR / "pass5_final_manifest.json")

In [ ]:
# ============================================================
# Cell 17 — ARTIFACT INTEGRITY + SCRATCH CLEANUP
# ============================================================

tee_print("=" * 80)
tee_print("PASS 5 v10 FINAL ARTIFACT INTEGRITY")
tee_print("=" * 80)

required = [
    "pass5_final_comparison.csv", "pass5_final_comparison.json",
    "pass5_per_class_comparison.csv", "pass5_tree_calibration_audit.csv" if not DRY_RUN else "pass5_frozen_protocol.json",
    "pass5_bootstrap_95ci.csv", "pass5_tree_difference_block_bootstrap.csv",
    "pass5_neural_difference_ci_status.csv" if not DRY_RUN else "pass5_final_manifest.json",
    "Pass5_paper_results.md", "Pass5_paper_paragraph.txt", "pass5_final_manifest.json",
    "lightgbm_baseline.joblib", "xgboost_baseline.json", "lightgbm_config.json", "lightgbm_training_attempts.json",
    "xgboost_config.json", "preprocessor.joblib", "pass5_frozen_protocol.json",
]
if not DRY_RUN:
    required += ["lightgbm_april_stacker.joblib", "xgboost_april_stacker.joblib", "pass5_hpo_trials.csv", "pass5_stacker_config.json"]
missing = [name for name in required if not (RUN_DIR / name).exists()]
assert not missing, "Missing expected final artifacts:\n" + "\n".join(missing)

with open(RUN_DIR / "pass5_final_manifest.json", "r", encoding="utf-8") as f:
    _manifest_check = json.load(f)
assert _manifest_check["n_classes"] == 50
assert _manifest_check["feature_dimension"] == 101
with open(RUN_DIR / "xgboost_baseline.json", "r", encoding="utf-8") as f:
    _xgb_bundle_check = json.load(f)
assert _xgb_bundle_check.get("model_type") == "xgboost_one_vs_rest_binary_ensemble"
assert int(_xgb_bundle_check.get("num_classes", -1)) == N_CLASSES
assert len(_xgb_bundle_check.get("model_files", [])) == N_CLASSES
_missing_xgb = [item.get("file", "<missing>") for item in _xgb_bundle_check["model_files"] if not (RUN_DIR / item.get("file", "__missing__")).exists()]
assert not _missing_xgb, "Missing XGBoost class models:\n" + "\n".join(_missing_xgb)

if not DRY_RUN:
    assert _manifest_check["train_rows"] == TRAIN_EXPECTED_ROWS
    assert _manifest_check["test"]["actual_flows"] == TEST_EXPECTED_ROWS
    assert int(LGBM_TEST_METRICS["samples"]) == TEST_EXPECTED_ROWS
    assert int(XGB_TEST_METRICS["samples"]) == TEST_EXPECTED_ROWS

# Remove scratch-only feature/label/weight arrays after all model, prediction,
# bootstrap and report artifacts have passed validation. Keep outputs needed to
# audit the experiment: per-flow tree predictions, day-level confusion matrices,
# model files, configs, HPO logs, stackers, reports and manifest.
for _name in (
    "X_train", "y_train", "X_val", "y_val", "train_sample_weight", "train_counts", "class_weights",
    "X_test", "y_test", "test_day_ids", "X_hpo_pilot", "y_hpo_pilot", "X_hpo_val", "y_hpo_val",
    "X_cal_fit", "y_cal_fit", "X_cal_audit", "y_cal_audit",
):
    globals().pop(_name, None)
gc.collect()
try:
    if SCRATCH_DIR.exists():
        shutil.rmtree(SCRATCH_DIR)
    for _dirname in ("xgb_external_memory_probe",):
        _path = RUN_DIR / _dirname
        if _path.exists():
            shutil.rmtree(_path)
    tee_print("Scratch arrays/cache and completed-model checkpoints removed:", SCRATCH_DIR)
except Exception as _exc:
    tee_print("Scratch cleanup warning (not a metric failure):", repr(_exc))

tee_print("Required artifacts: PASSED")
tee_print("Model bundle checks: PASSED")
tee_print("FULL RUN COMPLETE." if not DRY_RUN else "DRY RUN COMPLETE — numerical results are not paper-valid.")
tee_print("Output directory:", RUN_DIR)
tee_print("Run log:", LOG_PATH)
LOG_FILE.flush()
LOG_FILE.close()
LOG_FILE = None